# SigFlow-Sim v4 — Legacy notebook snapshot

> **Superseded:** use `SigFlow_v4_Research_Runner.ipynb` or `sigflow_v4_research.py`. The canonical runner contains the rate-limit, regime, HAR, multi-seed, multi-horizon, rolling-origin, calibration, diagnostic, and reproducibility improvements; this snapshot is retained only to preserve the earlier analysis.

This notebook preserves the existing SigFlow conditional-flow model and its objective, while making the experimental protocol deeper, configuration-driven, and suitable for honest generalisation tests.

## Experimental guarantees

- **Three fitting sections plus final tests:** training, validation, calibration, and two untouched future-date test cohorts.
- **Two final generalisation cohorts:** trained tickers on unseen days, and completely held-out tickers on the same unseen days.
- **Purging at every chronological boundary:** no target window crosses from an earlier fitting section into a later one.
- **Held-out tickers never influence fitting:** preprocessing, early stopping, calibration, regime thresholds, transition priors, and the transferable HAR fallback use training tickers only.
- **No backward filling:** market context is forward-filled for at most three sessions; incomplete samples are discarded.
- **Canonical feature matrix:** multiscale shape, amplitude, OHLC, joint-market, and lead-lag signatures are built once, then masks select matched ablations.
- **Sector-balanced transfer test:** 22 training and 22 held-out tickers cover the same 11 sectors.
- **Prespecified inference:** unseen-ticker QLIKE versus HAR is primary; tolerance accuracy, other baselines, sectors, and days are secondary.
- **Unchanged model mathematics:** the shared encoder, gate, quantile head, conditional flow experts, density/Jacobian, sampling equations, and full objective retain their original formulas.
- **Result-only reporting:** the notebook displays and saves test forecasts, errors, success flags, intervals, regime outputs, and summaries—not raw OHLCV or feature rows.

## Success definition

A forecast is successful when its median volatility forecast is within 20% of realised volatility. Each row is issued on `origin_date` and predicts realised volatility over the following `horizon` sessions, shown by `target_start_date` and `target_end_date`.

## First run

The default `gpu_long` profile is intended for a normal CUDA GPU, is deliberately allowed to run well beyond ten minutes, and uses a larger, diversified training universe plus completely held-out tickers. Runtime still depends on the GPU and available data. Use `gpu_balanced` for a shorter real-data run, or set `PROFILE_NAME = "smoke"` (or launch with `SIGFLOW_PROFILE=smoke`) for a short pipeline check.


In [ ]:
# Environment diagnostic — no package installation is attempted.

import importlib.util
import sys

REQUIRED_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "torch": "torch",
}

missing = [
    package
    for package, import_name in REQUIRED_IMPORTS.items()
    if importlib.util.find_spec(import_name) is None
]

print("Python executable:", sys.executable)
if missing:
    raise ModuleNotFoundError(
        "Missing core scientific packages: " + ", ".join(missing)
    )

print("Core packages are available.")
print("No yfinance, iisignature, or nflows installation is required.")

In [ ]:
from __future__ import annotations

import copy
import io
import json
import math
import os
import random
import re
import time
import urllib.parse
import urllib.request
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from IPython.display import HTML, display
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPS = 1e-8
REGIME_NAMES = np.array(["Contracting", "Stable", "Expanding"])

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Data route: cache → direct Yahoo → Stooq → optional synthetic fallback")


## Configuration

The next cell is the notebook control panel. The most useful controls are:

- `PROFILE_NAME`: `smoke`, `gpu_balanced`, `gpu_long`, or `deep`. The default `gpu_long` profile uses five seeds, at least 100 epochs per seed, and a 320-epoch cap.
- `training_tickers` and `unseen_test_tickers`: disjoint, sector-balanced 22-name universes.
- signature-family/window switches: shape and amplitude at 10/20/60 sessions plus OHLC, joint-market, and lead-lag paths.
- significance controls: primary cohort/baseline/metric, block sensitivities, practical effect size, and seed-consistency threshold.
- feature-family switches plus `disabled_feature_groups` and `disabled_feature_names`; every canonical feature can be removed by its exact printed name.
- `RUN_GROUP_ABLATIONS` and `RUN_INDIVIDUAL_FEATURE_ABLATIONS`. Ablations are averaged across expanding inner folds wholly inside training, so outer validation, calibration, and final test retain their roles.
- `DISPLAY_ALL_TEST_RESULTS`: renders every result row in a scrollable table; the same rows are saved to `detailed_test_results.csv`.

The default run is intentionally substantial and writes to its own `gpu_long_base` result directory. Use `smoke` to verify the whole pipeline before committing GPU time; runtime estimates are hardware-dependent.


In [ ]:
# ---------------------------------------------------------------------
# Primary controls: edit these, then use Restart and Run All.
# ---------------------------------------------------------------------
PROFILE_NAME = os.environ.get("SIGFLOW_PROFILE", "gpu_long").strip().lower()
RUN_GROUP_ABLATIONS = False
RUN_INDIVIDUAL_FEATURE_ABLATIONS = False
DISPLAY_ALL_TEST_RESULTS = True

SELECTED_GROUP_ABLATIONS = (
    "no_signature",
    "no_shape_signatures",
    "no_amplitude_signatures",
    "no_ohlc_signatures",
    "no_joint_market_signatures",
    "no_lead_lag_signatures",
    "no_signature_w10",
    "no_signature_w20",
    "no_signature_w60",
    "no_return_statistics",
    "no_ohlcv",
    "no_market",
    "no_regime_classification",
    "regime_weight_030",
    "absolute_regime_labels",
    "with_ticker_identity",
    "no_auxiliary_losses",
)

# Empty means every active feature when RUN_INDIVIDUAL_FEATURE_ABLATIONS=True.
# Otherwise enter exact feature names printed by the active-feature manifest.
INDIVIDUAL_FEATURES_TO_ABLATE: tuple[str, ...] = ()


TRAINING_TICKERS_BY_SECTOR = {
    "Information Technology": ("AAPL", "MSFT"),
    "Communication Services": ("GOOGL", "DIS"),
    "Consumer Discretionary": ("AMZN", "HD"),
    "Consumer Staples": ("PG", "KO"),
    "Financials": ("JPM", "BAC"),
    "Health Care": ("JNJ", "PFE"),
    "Industrials": ("CAT", "HON"),
    "Energy": ("XOM", "CVX"),
    "Materials": ("LIN", "APD"),
    "Utilities": ("NEE", "DUK"),
    "Real Estate": ("PLD", "AMT"),
}
UNSEEN_TEST_TICKERS_BY_SECTOR = {
    "Information Technology": ("NVDA", "AMD"),
    "Communication Services": ("META", "NFLX"),
    "Consumer Discretionary": ("TSLA", "NKE"),
    "Consumer Staples": ("PEP", "COST"),
    "Financials": ("GS", "MS"),
    "Health Care": ("UNH", "MRK"),
    "Industrials": ("GE", "UPS"),
    "Energy": ("COP", "SLB"),
    "Materials": ("FCX", "NEM"),
    "Utilities": ("AEP", "SO"),
    "Real Estate": ("O", "SPG"),
}


def flatten_sector_universe(
    universe: dict[str, tuple[str, ...]],
) -> tuple[str, ...]:
    return tuple(
        ticker
        for tickers in universe.values()
        for ticker in tickers
    )


TRAINING_TICKERS = flatten_sector_universe(TRAINING_TICKERS_BY_SECTOR)
UNSEEN_TEST_TICKERS = flatten_sector_universe(
    UNSEEN_TEST_TICKERS_BY_SECTOR
)
TICKER_SECTORS = {
    ticker: sector
    for universe in (
        TRAINING_TICKERS_BY_SECTOR,
        UNSEEN_TEST_TICKERS_BY_SECTOR,
    )
    for sector, tickers in universe.items()
    for ticker in tickers
}


PROFILE_SETTINGS = {
    "smoke": {
        "quick_mode": True,
        "training_tickers": ("AAPL", "MSFT", "JPM"),
        "unseen_test_tickers": ("TSLA", "AMD"),
        "use_real_market_data": False,
        "allow_synthetic_fallback": True,
        "start_date": "2020-01-01",
        "epochs": 2,
        "batch_size": 256,
        "hidden_size": 48,
        "patience": 2,
        "minimum_epochs": 1,
        "scheduler_patience": 1,
        "ensemble_seeds": (42,),
        "prediction_samples": 96,
        "bootstrap_repetitions": 50,
        "run_block_bootstrap": False,
        "minimum_test_origins_per_ticker": 60,
    },
    "gpu_balanced": {
        "quick_mode": False,
        "training_tickers": TRAINING_TICKERS,
        "unseen_test_tickers": UNSEEN_TEST_TICKERS,
        "use_real_market_data": True,
        "allow_synthetic_fallback": False,
        "start_date": "2012-01-01",
        "epochs": 150,
        "batch_size": 512,
        "hidden_size": 96,
        "patience": 18,
        "minimum_epochs": 40,
        "scheduler_patience": 8,
        "ensemble_seeds": (11, 22, 33),
        "prediction_samples": 768,
        "bootstrap_repetitions": 750,
        "run_block_bootstrap": True,
        "minimum_test_origins_per_ticker": 400,
    },
    "gpu_long": {
        "quick_mode": False,
        "training_tickers": TRAINING_TICKERS,
        "unseen_test_tickers": UNSEEN_TEST_TICKERS,
        "use_real_market_data": True,
        "allow_synthetic_fallback": False,
        "start_date": "2010-01-01",
        "epochs": 320,
        "batch_size": 256,
        "hidden_size": 128,
        "patience": 40,
        "minimum_epochs": 100,
        "scheduler_patience": 8,
        "ensemble_seeds": (11, 22, 33, 44, 55),
        "prediction_samples": 1536,
        "bootstrap_repetitions": 2000,
        "run_block_bootstrap": True,
        "minimum_test_origins_per_ticker": 500,
    },
    "deep": {
        "quick_mode": False,
        "training_tickers": TRAINING_TICKERS,
        "unseen_test_tickers": UNSEEN_TEST_TICKERS,
        "use_real_market_data": True,
        "allow_synthetic_fallback": False,
        "start_date": "2010-01-01",
        "epochs": 480,
        "batch_size": 256,
        "hidden_size": 160,
        "patience": 55,
        "minimum_epochs": 150,
        "scheduler_patience": 10,
        "ensemble_seeds": (11, 22, 33, 44, 55, 66, 77),
        "prediction_samples": 2048,
        "bootstrap_repetitions": 3000,
        "run_block_bootstrap": True,
        "minimum_test_origins_per_ticker": 500,
    },
}

if PROFILE_NAME not in PROFILE_SETTINGS:
    raise ValueError(
        f"Unknown PROFILE_NAME={PROFILE_NAME!r}; choose from "
        f"{tuple(PROFILE_SETTINGS)}."
    )

PROFILE = PROFILE_SETTINGS[PROFILE_NAME]


@dataclass(frozen=True)
class AblationSpec:
    name: str
    description: str
    overrides: tuple[tuple[str, object], ...]


@dataclass(frozen=True)
class Config:
    profile_name: str = PROFILE_NAME
    quick_mode: bool = bool(PROFILE["quick_mode"])
    experiment_mode: str = (
        "pipeline_test" if PROFILE["quick_mode"] else "real_experiment"
    )
    run_name: str = f"{PROFILE_NAME}_base"

    # Target universes. Held-out tickers are never used for fitting.
    training_tickers: tuple[str, ...] = PROFILE["training_tickers"]
    unseen_test_tickers: tuple[str, ...] = PROFILE["unseen_test_tickers"]

    # Data and provenance
    use_real_market_data: bool = bool(PROFILE["use_real_market_data"])
    allow_synthetic_fallback: bool = bool(PROFILE["allow_synthetic_fallback"])
    refresh_data: bool = False
    network_preflight_timeout_seconds: int = 3
    network_request_timeout_seconds: int = 8
    run_network_preflight: bool = True
    market_symbols: tuple[str, ...] = ("SPY", "QQQ", "^VIX")
    include_market_context: bool = True
    start_date: str = str(PROFILE["start_date"])
    end_date: str = "2026-07-25"
    cache_dir: str = "sigflow_v4_cache"
    output_root: str = "sigflow_v4_outputs"

    # Forecast construction and configurable signature lifts.
    window: int = 60
    horizon: int = 10
    annualisation: float = 252.0
    ewma_lambda: float = 0.94
    signature_windows: tuple[int, ...] = (10, 20, 60)
    secondary_signature_windows: tuple[int, ...] = (20,)
    signature_depth: int = 3
    short_signature_depth: int = 2
    amplitude_signature_depth: int = 2
    ohlc_signature_depth: int = 2
    joint_market_signature_depth: int = 2
    lead_lag_signature_depth: int = 2
    signature_return_scale: float = 0.01
    signature_vix_change_scale: float = 0.05
    joint_signature_symbols: tuple[str, str] = ("SPY", "^VIX")
    market_forward_fill_limit: int = 3

    # Model-input feature switches. The canonical feature matrix is still
    # built once, so ablations share exactly the same rows and values.
    include_signature_features: bool = True
    include_shape_signatures: bool = True
    include_amplitude_signatures: bool = True
    include_ohlc_signatures: bool = True
    include_joint_market_signatures: bool = True
    include_lead_lag_signatures: bool = True
    include_return_statistics: bool = True
    include_ohlcv_features: bool = True
    include_market_features: bool = True
    include_ticker_identity: bool = False
    disabled_feature_groups: tuple[str, ...] = ()
    disabled_feature_names: tuple[str, ...] = ()

    # Strict chronological sections, calculated from training tickers only.
    train_fraction: float = 0.65
    validation_fraction: float = 0.10
    calibration_fraction: float = 0.10
    inner_validation_folds: int = 3
    inner_validation_fraction_of_training: float = 0.30

    # Training-only preprocessing and transferable regime definitions.
    winsor_lower_quantile: float = 0.001
    winsor_upper_quantile: float = 0.999
    standardised_clip: float = 8.0
    regime_threshold_mode: str = "pooled_training"
    regime_target_mode: str = "relative_to_recent_volatility"
    ticker_specific_regime_smoothing: bool = False

    # Model capacity (the conditional-flow equations are unchanged).
    regimes: int = 3
    hidden_size: int = int(PROFILE["hidden_size"])
    dropout: float = 0.20
    minimum_scale: float = 0.03
    maximum_scale: float = 1.50
    maximum_skew: float = 1.50
    minimum_tail: float = 0.70
    maximum_tail: float = 2.00

    # Original full objective and its independently ablatable terms.
    expert_alignment_weight: float = 0.30
    regime_classification_weight: float = 0.10
    medium_class_multiplier: float = 1.00
    gate_balance_weight: float = 0.005
    qlike_weight: float = 0.05
    quantile_weight: float = 0.10
    label_smoothing: float = 0.02

    # Deeper normal-GPU optimisation profile.
    epochs: int = int(PROFILE["epochs"])
    batch_size: int = int(PROFILE["batch_size"])
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    gradient_clip: float = 1.0
    patience: int = int(PROFILE["patience"])
    minimum_epochs: int = int(PROFILE["minimum_epochs"])
    early_stopping_min_delta: float = 1e-4
    scheduler_patience: int = int(PROFILE["scheduler_patience"])
    scheduler_factor: float = 0.50
    scheduler_cooldown: int = 2
    minimum_learning_rate: float = 2.5e-6
    checkpoint_metric: str = "validation_qlike"
    print_every: int = 1 if PROFILE["quick_mode"] else 5
    pin_memory: bool = True
    matmul_precision: str = "high"

    # Ensemble, deterministic evaluation sampling, and predictions.
    ensemble_seeds: tuple[int, ...] = PROFILE["ensemble_seeds"]
    prediction_seed: int = 20260803
    prediction_samples: int = int(PROFILE["prediction_samples"])
    prediction_batch_size: int = 512
    minimum_reported_volatility: float = 0.01
    maximum_reported_volatility: float = 5.00

    # Calibration-only searches. Toggle these manually for a prespecified
    # final-protocol sensitivity run; validation feature ablations do not
    # pretend to score calibration that has deliberately not been fitted.
    calibrate_intervals: bool = True
    calibrate_regime_probabilities: bool = True
    interval_scale_min: float = 0.50
    interval_scale_max: float = 2.00
    interval_scale_candidates: int = 76
    temperature_min: float = 0.25
    temperature_max: float = 4.00
    temperature_candidates: int = 151

    # Result accuracy and dependence-aware evaluation.
    success_relative_tolerances: tuple[float, ...] = (
        0.05, 0.10, 0.20, 0.30, 0.50,
    )
    primary_success_tolerance: float = 0.20
    minimum_test_origins_per_ticker: int = int(
        PROFILE["minimum_test_origins_per_ticker"]
    )
    primary_evaluation_cohort: str = "unseen_ticker_unseen_day"
    primary_baseline: str = "HAR-style transferable"
    primary_metric: str = "qlike"
    primary_min_relative_improvement: float = 0.05
    minimum_seed_improvement_fraction: float = 0.80
    significance_alpha: float = 0.05
    run_block_bootstrap: bool = bool(PROFILE["run_block_bootstrap"])
    bootstrap_repetitions: int = int(PROFILE["bootstrap_repetitions"])
    bootstrap_sensitivity_repetitions: int = 500
    bootstrap_block_days: int = 20
    bootstrap_block_sensitivity_days: tuple[int, ...] = (10, 20, 40, 60)
    ticker_date_bootstrap_repetitions: int = 1000
    display_all_test_results: bool = DISPLAY_ALL_TEST_RESULTS
    plot_each_ticker: bool = False
    save_audit_artifacts: bool = False

    # Development-only ablation budget. Set reduced=False in the runner
    # for fully matched budgets after the shortlist is stable.
    ablation_reduced_budget: bool = True
    ablation_epochs: int = 60
    ablation_patience: int = 8
    ablation_ensemble_seeds: tuple[int, ...] = (42,)

    seed: int = 42

    @property
    def target_tickers(self) -> tuple[str, ...]:
        return self.training_tickers + self.unseen_test_tickers

    @property
    def output_dir(self) -> str:
        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", self.run_name)
        return str(Path(self.output_root) / safe_name)


GROUP_ABLATIONS = {
    spec.name: spec
    for spec in (
        AblationSpec("no_signature", "Remove every path-signature coordinate.", (("include_signature_features", False),)),
        AblationSpec("no_signature_l3", "Remove signature level 3 only.", (("disabled_feature_groups", ("signature_l3",)),)),
        AblationSpec("no_shape_signatures", "Remove locally normalised shape signatures.", (("include_shape_signatures", False),)),
        AblationSpec("no_amplitude_signatures", "Remove scale-preserving amplitude signatures.", (("include_amplitude_signatures", False),)),
        AblationSpec("no_ohlc_signatures", "Remove OHLC range/gap path signatures.", (("include_ohlc_signatures", False),)),
        AblationSpec("no_joint_market_signatures", "Remove joint asset-SPY-VIX signatures.", (("include_joint_market_signatures", False),)),
        AblationSpec("no_lead_lag_signatures", "Remove lead-lag signatures.", (("include_lead_lag_signatures", False),)),
        AblationSpec("no_signature_w10", "Remove every 10-session signature block.", (("disabled_feature_groups", ("signature_w10",)),)),
        AblationSpec("no_signature_w20", "Remove every 20-session signature block.", (("disabled_feature_groups", ("signature_w20",)),)),
        AblationSpec("no_signature_w60", "Remove every 60-session signature block.", (("disabled_feature_groups", ("signature_w60",)),)),
        AblationSpec("no_return_statistics", "Remove all return-statistic features.", (("include_return_statistics", False),)),
        AblationSpec("no_ohlcv", "Remove all OHLCV-derived features.", (("include_ohlcv_features", False),)),
        AblationSpec("no_market", "Remove all SPY, QQQ, and VIX context features.", (("include_market_features", False),)),
        AblationSpec("no_spy", "Remove SPY context only.", (("disabled_feature_groups", ("market_spy",)),)),
        AblationSpec("no_qqq", "Remove QQQ context only.", (("disabled_feature_groups", ("market_qqq",)),)),
        AblationSpec("no_vix", "Remove VIX context only.", (("disabled_feature_groups", ("market_vix",)),)),
        AblationSpec("with_ticker_identity", "Add seen-ticker identity; unseen tickers receive a neutral all-zero code.", (("include_ticker_identity", True),)),
        AblationSpec("no_expert_alignment", "Set the expert-alignment weight to zero.", (("expert_alignment_weight", 0.0),)),
        AblationSpec("no_regime_classification", "Set the regime-classification weight to zero.", (("regime_classification_weight", 0.0),)),
        AblationSpec("regime_weight_030", "Restore the stronger 0.30 classification weight.", (("regime_classification_weight", 0.30),)),
        AblationSpec("absolute_regime_labels", "Use pooled absolute-volatility regime labels.", (("regime_target_mode", "absolute_volatility"),)),
        AblationSpec("no_gate_balance", "Set the gate-balance weight to zero.", (("gate_balance_weight", 0.0),)),
        AblationSpec("no_qlike_term", "Set the QLIKE auxiliary weight to zero.", (("qlike_weight", 0.0),)),
        AblationSpec("no_quantile_term", "Set the quantile auxiliary weight to zero.", (("quantile_weight", 0.0),)),
        AblationSpec("no_auxiliary_losses", "Train with mixture NLL only.", (
            ("expert_alignment_weight", 0.0),
            ("regime_classification_weight", 0.0),
            ("gate_balance_weight", 0.0),
            ("qlike_weight", 0.0),
            ("quantile_weight", 0.0),
        )),
    )
}


def validate_config(cfg: Config) -> None:
    if not cfg.training_tickers or not cfg.unseen_test_tickers:
        raise ValueError("Both training and unseen-test ticker lists must be non-empty.")
    if len(set(cfg.target_tickers)) != len(cfg.target_tickers):
        raise ValueError("Training and unseen-test tickers must be unique and disjoint.")
    if set(cfg.target_tickers) & set(cfg.market_symbols):
        raise ValueError("Target tickers and market-context symbols must be disjoint.")
    if cfg.train_fraction + cfg.validation_fraction + cfg.calibration_fraction >= 1.0:
        raise ValueError("Train, validation, and calibration fractions must sum below 1.")
    if cfg.regime_threshold_mode not in {"pooled_training", "per_training_ticker"}:
        raise ValueError("regime_threshold_mode must be pooled_training or per_training_ticker.")
    if cfg.regime_target_mode not in {
        "relative_to_recent_volatility", "absolute_volatility"
    }:
        raise ValueError("Unknown regime_target_mode.")
    if tuple(sorted(set(cfg.signature_windows))) != cfg.signature_windows:
        raise ValueError("signature_windows must be sorted and unique.")
    if tuple(sorted(set(cfg.secondary_signature_windows))) != cfg.secondary_signature_windows:
        raise ValueError("secondary_signature_windows must be sorted and unique.")
    if not cfg.signature_windows or max(cfg.signature_windows) > cfg.window:
        raise ValueError("Signature windows must be non-empty and at most window.")
    if not set(cfg.secondary_signature_windows) <= set(cfg.signature_windows):
        raise ValueError("Secondary signature windows must be signature windows.")
    signature_depths = (
        cfg.signature_depth,
        cfg.short_signature_depth,
        cfg.amplitude_signature_depth,
        cfg.ohlc_signature_depth,
        cfg.joint_market_signature_depth,
        cfg.lead_lag_signature_depth,
    )
    if any(depth not in (1, 2, 3) for depth in signature_depths):
        raise ValueError("Every signature depth must be 1, 2, or 3.")
    if cfg.signature_return_scale <= 0.0 or cfg.signature_vix_change_scale <= 0.0:
        raise ValueError("Signature scales must be positive.")
    if not set(cfg.joint_signature_symbols) <= set(cfg.market_symbols):
        raise ValueError("Joint-signature symbols must be configured market symbols.")
    if not set(cfg.target_tickers) <= set(TICKER_SECTORS):
        raise ValueError("Every configured target ticker needs exactly one sector.")
    if cfg.experiment_mode == "real_experiment":
        training_sectors = {
            TICKER_SECTORS[ticker] for ticker in cfg.training_tickers
        }
        unseen_sectors = {
            TICKER_SECTORS[ticker] for ticker in cfg.unseen_test_tickers
        }
        if training_sectors != unseen_sectors or len(unseen_sectors) < 10:
            raise ValueError(
                "Training and unseen universes must cover the same sectors."
            )
        if len(cfg.unseen_test_tickers) < 15:
            raise ValueError(
                "A real cross-ticker claim requires at least 15 held-out tickers."
            )
    if cfg.checkpoint_metric not in {"validation_objective", "validation_qlike"}:
        raise ValueError("checkpoint_metric must be validation_objective or validation_qlike.")
    required_tolerances = {0.05, 0.10, 0.20, 0.30, 0.50}
    if not required_tolerances <= set(cfg.success_relative_tolerances):
        raise ValueError(
            "success_relative_tolerances must include 5%, 10%, 20%, 30%, and 50%."
        )
    if cfg.primary_success_tolerance not in cfg.success_relative_tolerances:
        raise ValueError(
            "primary_success_tolerance must appear in success_relative_tolerances."
        )
    if cfg.bootstrap_block_days not in cfg.bootstrap_block_sensitivity_days:
        raise ValueError("The primary bootstrap block must appear in sensitivity blocks.")
    if any(days < cfg.horizon for days in cfg.bootstrap_block_sensitivity_days):
        raise ValueError("Bootstrap blocks cannot be shorter than the forecast horizon.")
    if not 0.0 < cfg.primary_min_relative_improvement < 1.0:
        raise ValueError("primary_min_relative_improvement must be in (0, 1).")
    if not 0.0 < cfg.minimum_seed_improvement_fraction <= 1.0:
        raise ValueError("minimum_seed_improvement_fraction must be in (0, 1].")
    if not 1 <= cfg.minimum_epochs <= cfg.epochs:
        raise ValueError("minimum_epochs must be between 1 and epochs.")
    if not 1 <= cfg.scheduler_patience < cfg.patience:
        raise ValueError("scheduler_patience must be positive and below patience.")
    if not 0.0 < cfg.scheduler_factor < 1.0:
        raise ValueError("scheduler_factor must be strictly between zero and one.")
    if not 0.0 < cfg.minimum_learning_rate < cfg.learning_rate:
        raise ValueError("minimum_learning_rate must be positive and below learning_rate.")
    if cfg.scheduler_cooldown < 0 or cfg.early_stopping_min_delta < 0.0:
        raise ValueError("Scheduler cooldown and stopping delta cannot be negative.")
    if cfg.experiment_mode == "real_experiment" and cfg.allow_synthetic_fallback:
        raise ValueError("Synthetic fallback must be disabled for a final real experiment.")


CFG = Config()
validate_config(CFG)
print(json.dumps({**asdict(CFG), "target_tickers": CFG.target_tickers, "output_dir": CFG.output_dir}, indent=2))


## Reproducibility, robust preprocessing, and data containers

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        pass

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False


@dataclass
class RobustStandardiser:
    lower: np.ndarray
    upper: np.ndarray
    mean: np.ndarray
    scale: np.ndarray
    z_clip: float

    @classmethod
    def fit(
        cls,
        values: np.ndarray,
        lower_quantile: float,
        upper_quantile: float,
        z_clip: float,
    ) -> "RobustStandardiser":
        lower = np.nanquantile(values, lower_quantile, axis=0)
        upper = np.nanquantile(values, upper_quantile, axis=0)
        upper = np.maximum(upper, lower)

        clipped = np.clip(values, lower, upper)
        mean = np.nanmean(clipped, axis=0)
        scale = np.nanstd(clipped, axis=0)
        scale = np.where(
            np.isfinite(scale) & (scale > 1e-6),
            scale,
            1.0,
        )

        return cls(
            lower=lower.astype(np.float32),
            upper=upper.astype(np.float32),
            mean=mean.astype(np.float32),
            scale=scale.astype(np.float32),
            z_clip=float(z_clip),
        )

    def transform(self, values: np.ndarray) -> np.ndarray:
        clipped = np.clip(values, self.lower, self.upper)
        transformed = (clipped - self.mean) / self.scale
        transformed = np.clip(
            transformed,
            -self.z_clip,
            self.z_clip,
        )
        return np.nan_to_num(
            transformed,
            nan=0.0,
            posinf=self.z_clip,
            neginf=-self.z_clip,
        ).astype(np.float32)


@dataclass
class MarketDataset:
    features: np.ndarray
    targets_log_vol: np.ndarray
    metadata: pd.DataFrame
    feature_names: list[str]
    data_source: str
    available_market_symbols: tuple[str, ...]
    data_manifest: pd.DataFrame
    skipped_samples: pd.DataFrame


@dataclass
class PreparedSplit:
    train_indices: np.ndarray
    validation_indices: np.ndarray
    calibration_indices: np.ndarray
    test_indices: np.ndarray
    seen_ticker_unseen_day_indices: np.ndarray
    unseen_ticker_unseen_day_indices: np.ndarray
    context: np.ndarray
    regime_labels: np.ndarray
    ticker_thresholds: np.ndarray
    class_weights: np.ndarray
    train_regime_proportions: np.ndarray
    transition_matrices: np.ndarray
    initial_regime_probabilities: np.ndarray
    standardiser: RobustStandardiser
    active_feature_indices: np.ndarray
    active_feature_names: list[str]
    dropped_feature_names: list[str]
    context_feature_names: list[str]
    train_cutoff: pd.Timestamp
    validation_cutoff: pd.Timestamp
    calibration_cutoff: pd.Timestamp
    leakage_audit: pd.DataFrame


set_seed(CFG.seed)


## Multiscale, amplitude-preserving path signatures

The canonical representation separates **shape** from **amplitude**. Shape paths retain local standardisation; amplitude paths use a fixed 1% return unit so volatility scale is not erased. Signatures are generated over 10, 20, and 60 sessions, with additional 20-session OHLC range/gap, joint asset–SPY–VIX, and lead-lag paths. Short and secondary paths use depth two for ordinary-GPU practicality; the 60-session shape path retains depth three.

Every family and window has a named zero-mask ablation. The full signature tensor is dependency-free and validated against amplitude/shape invariants during testing.


In [ ]:
def truncated_signature(path: np.ndarray, depth: int = 3) -> np.ndarray:
    if depth not in (1, 2, 3):
        raise ValueError("Supported signature depths are 1, 2, and 3.")

    increments = np.diff(np.asarray(path, dtype=np.float64), axis=0)
    dimension = path.shape[1]

    level1 = np.zeros(dimension, dtype=np.float64)
    level2 = np.zeros((dimension, dimension), dtype=np.float64)
    level3 = np.zeros(
        (dimension, dimension, dimension),
        dtype=np.float64,
    )

    for increment in increments:
        segment1 = increment
        old1 = level1.copy()
        old2 = level2.copy()
        segment2 = None

        if depth >= 2:
            segment2 = np.einsum(
                "i,j->ij", increment, increment
            ) / 2.0
        if depth >= 3:
            segment3 = np.einsum(
                "i,j,k->ijk",
                increment,
                increment,
                increment,
            ) / 6.0

        level1 = old1 + segment1
        if depth >= 2 and segment2 is not None:
            level2 = (
                old2
                + np.einsum("i,j->ij", old1, segment1)
                + segment2
            )
        if depth >= 3 and segment2 is not None:
            level3 = (
                level3
                + np.einsum("ij,k->ijk", old2, segment1)
                + np.einsum("i,jk->ijk", old1, segment2)
                + segment3
            )

    levels = [level1.ravel()]
    if depth >= 2:
        levels.append(level2.ravel())
    if depth >= 3:
        levels.append(level3.ravel())
    return np.concatenate(levels)


def signature_feature_names(
    prefix: str,
    path_dimension: int,
    depth: int,
) -> list[str]:
    names = []
    for level in range(1, depth + 1):
        for index in range(path_dimension**level):
            names.append(f"signature_{prefix}_L{level}_{index}")
    return names

@dataclass(frozen=True)
class SignatureBlockSpec:
    family: str
    window: int
    path_dimension: int
    depth: int

    @property
    def prefix(self) -> str:
        return f"{self.family}_w{self.window}"


## Direct OHLCV loading with provenance and no invented prices

In [ ]:
REQUIRED_PRICE_COLUMNS = ["Open", "High", "Low", "Close"]


def clean_ohlcv(frame: pd.DataFrame, symbol: str) -> pd.DataFrame:
    frame = frame.copy()
    frame.index = pd.to_datetime(frame.index, errors="coerce")
    if getattr(frame.index, "tz", None) is not None:
        frame.index = frame.index.tz_localize(None)

    frame = frame[~frame.index.isna()]
    for column in REQUIRED_PRICE_COLUMNS + ["Volume"]:
        if column not in frame:
            frame[column] = np.nan
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    # Never replace missing Open/High/Low with Close. Those rows are dropped.
    frame = frame.dropna(subset=REQUIRED_PRICE_COLUMNS)
    frame = frame[
        (frame["Open"] > 0)
        & (frame["High"] > 0)
        & (frame["Low"] > 0)
        & (frame["Close"] > 0)
    ]

    current_volume_missing = frame["Volume"].isna()
    if "VolumeMissing" in frame:
        cached_volume_missing = pd.to_numeric(
            frame["VolumeMissing"], errors="coerce"
        ).fillna(0.0).gt(0.0)
    else:
        cached_volume_missing = pd.Series(False, index=frame.index)
    frame["VolumeMissing"] = (
        current_volume_missing | cached_volume_missing
    ).astype(float)
    frame["Volume"] = frame["Volume"].fillna(0.0)

    frame = frame[
        ["Open", "High", "Low", "Close", "Volume", "VolumeMissing"]
    ]
    frame = frame.replace([np.inf, -np.inf], np.nan)
    frame = frame.dropna(subset=REQUIRED_PRICE_COLUMNS)
    frame = frame[~frame.index.duplicated(keep="last")].sort_index()

    if frame.empty:
        raise ValueError(f"No usable OHLCV observations for {symbol}.")
    return frame


def request_bytes(
    url: str,
    timeout: int,
) -> bytes:
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 Chrome/124 Safari/537.36"
            ),
            "Accept": "application/json,text/csv,*/*",
        },
    )
    with urllib.request.urlopen(
        request,
        timeout=timeout,
    ) as response:
        return response.read()


def network_preflight(cfg: Config) -> tuple[bool, str]:
    if not cfg.run_network_preflight:
        return True, "preflight disabled"

    probes = (
        (
            "Yahoo",
            "https://query1.finance.yahoo.com/v8/finance/chart/"
            "AAPL?range=5d&interval=1d",
        ),
        (
            "Stooq",
            "https://stooq.com/q/d/l/?s=aapl.us&i=d",
        ),
    )
    print(
        "Checking external market-data connectivity "
        f"(timeout={cfg.network_preflight_timeout_seconds}s)...",
        flush=True,
    )

    failures = []
    for provider, url in probes:
        try:
            payload = request_bytes(
                url,
                timeout=cfg.network_preflight_timeout_seconds,
            )
            if not payload:
                raise ValueError("empty response")
            print(
                f"  {provider} connectivity check succeeded.",
                flush=True,
            )
            return True, f"{provider} reachable"
        except Exception as exc:
            message = f"{provider}: {type(exc).__name__}: {exc}"
            failures.append(message)
            print(
                "  Connectivity check failed quickly: " + message,
                flush=True,
            )
    return False, " | ".join(failures)


def download_yahoo_frame(
    symbol: str,
    start_date: str,
    end_date: str,
    timeout: int,
) -> pd.DataFrame:
    period1 = int(pd.Timestamp(start_date, tz="UTC").timestamp())
    period2 = int(pd.Timestamp(end_date, tz="UTC").timestamp())

    parameters = urllib.parse.urlencode({
        "period1": period1,
        "period2": period2,
        "interval": "1d",
        "events": "history",
        "includeAdjustedClose": "true",
    })
    quoted_symbol = urllib.parse.quote(symbol, safe="")
    url = (
        "https://query1.finance.yahoo.com/v8/finance/chart/"
        f"{quoted_symbol}?{parameters}"
    )

    payload = json.loads(
        request_bytes(url, timeout=timeout).decode("utf-8")
    )
    chart = payload.get("chart", {})
    if chart.get("error"):
        raise RuntimeError(str(chart["error"]))

    results = chart.get("result")
    if not results:
        raise ValueError(f"Yahoo returned no result for {symbol}.")

    result = results[0]
    timestamps = result.get("timestamp") or []
    quote_blocks = result.get("indicators", {}).get("quote") or []
    if not timestamps or not quote_blocks:
        raise ValueError(f"Yahoo returned incomplete OHLCV for {symbol}.")

    quote = quote_blocks[0]
    dates = pd.to_datetime(
        timestamps,
        unit="s",
        utc=True,
    ).tz_localize(None)

    frame = pd.DataFrame({
        "Open": quote.get("open"),
        "High": quote.get("high"),
        "Low": quote.get("low"),
        "Close": quote.get("close"),
        "Volume": quote.get("volume"),
    }, index=dates)

    adjusted_blocks = (
        result.get("indicators", {}).get("adjclose") or []
    )
    adjusted = (
        adjusted_blocks[0].get("adjclose")
        if adjusted_blocks else None
    )

    if adjusted is not None:
        raw_close = pd.to_numeric(
            frame["Close"], errors="coerce"
        )
        adjustment = (
            pd.Series(adjusted, index=dates) / raw_close
        )
        adjustment = adjustment.replace(
            [np.inf, -np.inf], np.nan
        )

        for column in REQUIRED_PRICE_COLUMNS:
            frame[column] = (
                pd.to_numeric(frame[column], errors="coerce")
                * adjustment
            )

    return clean_ohlcv(frame, symbol)


def stooq_symbol(symbol: str) -> str:
    if symbol == "^VIX":
        return "vix"
    return symbol.lower().replace(".", "-") + ".us"


def download_stooq_frame(
    symbol: str,
    start_date: str,
    end_date: str,
    timeout: int,
) -> pd.DataFrame:
    parameters = urllib.parse.urlencode({
        "s": stooq_symbol(symbol),
        "d1": pd.Timestamp(start_date).strftime("%Y%m%d"),
        "d2": pd.Timestamp(end_date).strftime("%Y%m%d"),
        "i": "d",
    })
    raw = request_bytes(
        f"https://stooq.com/q/d/l/?{parameters}",
        timeout=timeout,
    )
    frame = pd.read_csv(io.BytesIO(raw))

    if frame.empty or "Date" not in frame or "Close" not in frame:
        preview = raw[:150].decode("utf-8", errors="replace")
        raise ValueError(f"Stooq returned no useful data: {preview!r}")

    frame.index = pd.to_datetime(frame.pop("Date"), errors="coerce")
    return clean_ohlcv(frame, symbol)


def synthetic_frames(cfg: Config) -> dict[str, pd.DataFrame]:
    dates = pd.date_range(
        cfg.start_date,
        cfg.end_date,
        freq="B",
        inclusive="left",
    )
    rng = np.random.default_rng(cfg.seed + 9000)

    regimes = np.zeros(len(dates), dtype=int)
    transition = np.array([
        [0.975, 0.023, 0.002],
        [0.030, 0.940, 0.030],
        [0.010, 0.080, 0.910],
    ])
    for index in range(1, len(dates)):
        regimes[index] = rng.choice(
            3,
            p=transition[regimes[index - 1]],
        )

    common_volatility = np.array(
        [0.007, 0.014, 0.030]
    )[regimes]
    market_return = rng.normal(0.0002, common_volatility)

    frames = {}
    symbols = list(cfg.target_tickers) + (
        list(cfg.market_symbols)
        if cfg.include_market_context else []
    )

    for symbol_id, symbol in enumerate(symbols):
        local_rng = np.random.default_rng(
            cfg.seed + 1000 * (symbol_id + 1)
        )

        if symbol == "^VIX":
            close = (
                13.0
                + 7.0 * regimes
                + 250.0 * np.abs(market_return)
                + local_rng.normal(0.0, 1.4, len(dates))
            )
            close = np.maximum(close, 8.0)
            open_price = close * np.exp(
                local_rng.normal(0.0, 0.01, len(dates))
            )
        else:
            beta = 0.75 + 0.12 * symbol_id
            idiosyncratic = local_rng.normal(
                0.0001,
                np.array([0.006, 0.010, 0.020])[regimes],
            )
            returns = beta * market_return + idiosyncratic
            close = 100.0 * np.exp(np.cumsum(returns))
            previous_close = np.concatenate([[close[0]], close[:-1]])
            open_price = previous_close * np.exp(
                local_rng.normal(
                    0.0,
                    common_volatility * 0.25,
                )
            )

        range_scale = (
            0.004
            + 0.8 * np.abs(
                np.log(close / np.maximum(open_price, EPS))
            )
            + 0.5 * common_volatility
        )
        high = np.maximum(open_price, close) * np.exp(
            range_scale / 2.0
        )
        low = np.minimum(open_price, close) * np.exp(
            -range_scale / 2.0
        )
        volume = local_rng.lognormal(
            mean=16.0 + 7.0 * common_volatility,
            sigma=0.35,
            size=len(dates),
        )

        frames[symbol] = clean_ohlcv(
            pd.DataFrame({
                "Open": open_price,
                "High": high,
                "Low": low,
                "Close": close,
                "Volume": volume,
            }, index=dates),
            symbol,
        )

    return frames


def cache_path_for(symbol: str, cfg: Config) -> Path:
    safe_symbol = (
        symbol.replace("^", "INDEX_").replace("/", "_")
    )
    directory = Path(cfg.cache_dir)
    directory.mkdir(parents=True, exist_ok=True)
    return directory / (
        f"{safe_symbol}_{cfg.start_date}_{cfg.end_date}.csv"
    )


def load_cached_frame(
    symbol: str,
    cfg: Config,
) -> pd.DataFrame | None:
    path = cache_path_for(symbol, cfg)
    if not path.exists() or cfg.refresh_data:
        return None

    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    if frame.empty:
        return None

    print(f"  {symbol}: loaded cache")
    return clean_ohlcv(frame, symbol)


def download_symbol(
    symbol: str,
    cfg: Config,
) -> tuple[pd.DataFrame, str]:
    cached = load_cached_frame(symbol, cfg)
    if cached is not None:
        return cached, "cache"

    errors = []
    providers = [
        ("yahoo_direct", download_yahoo_frame),
        ("stooq_direct", download_stooq_frame),
    ]

    for provider, downloader in providers:
        print(
            f"  {symbol}: trying {provider} "
            f"(timeout={cfg.network_request_timeout_seconds}s)...",
            flush=True,
        )
        try:
            frame = downloader(
                symbol,
                cfg.start_date,
                cfg.end_date,
                cfg.network_request_timeout_seconds,
            )
            frame.to_csv(cache_path_for(symbol, cfg))
            print(
                f"  {symbol}: {len(frame):,} rows via {provider}",
                flush=True,
            )
            return frame, provider
        except Exception as exc:
            message = (
                f"{provider}: {type(exc).__name__}: {exc}"
            )
            errors.append(message)
            print(f"  {symbol}: {message}", flush=True)

    raise RuntimeError(" | ".join(errors))


def build_manifest(
    frames: dict[str, pd.DataFrame],
    providers: dict[str, str],
) -> pd.DataFrame:
    rows = []
    for symbol, frame in frames.items():
        rows.append({
            "symbol": symbol,
            "provider": providers.get(symbol, "unknown"),
            "rows": len(frame),
            "first_date": frame.index.min(),
            "last_date": frame.index.max(),
            "missing_volume_rows": int(
                frame["VolumeMissing"].sum()
            ),
        })
    return pd.DataFrame(rows)


def load_all_frames(
    cfg: Config,
) -> tuple[
    dict[str, pd.DataFrame],
    tuple[str, ...],
    str,
    pd.DataFrame,
]:
    if not cfg.use_real_market_data:
        print(
            "Real market-data requests are disabled; "
            "using synthetic pipeline-test data immediately.",
            flush=True,
        )
        frames = synthetic_frames(cfg)
        providers = {
            symbol: "synthetic_requested"
            for symbol in frames
        }
        return (
            frames,
            tuple(
                cfg.market_symbols
                if cfg.include_market_context else ()
            ),
            "synthetic_requested",
            build_manifest(frames, providers),
        )

    # Cached target data can be used without an internet connection.
    target_cache_complete = all(
        cache_path_for(symbol, cfg).exists()
        and not cfg.refresh_data
        for symbol in cfg.target_tickers
    )

    if not target_cache_complete:
        connected, reason = network_preflight(cfg)
        if not connected:
            if not cfg.allow_synthetic_fallback:
                raise ConnectionError(
                    "The runtime cannot reach the market-data endpoint. "
                    "Real-experiment mode will not substitute synthetic data. "
                    f"Preflight result: {reason}"
                )

            print(
                "WARNING: external internet is unavailable. "
                "Switching immediately to the clearly labelled "
                "synthetic pipeline-test dataset.",
                flush=True,
            )
            frames = synthetic_frames(cfg)
            providers = {
                symbol: "synthetic_fallback_fast"
                for symbol in frames
            }
            return (
                frames,
                tuple(
                cfg.market_symbols
                if cfg.include_market_context else ()
            ),
                "synthetic_fallback_fast",
                build_manifest(frames, providers),
            )
    else:
        print(
            "Target caches are available; skipping connectivity preflight.",
            flush=True,
        )

    frames = {}
    providers = {}

    try:
        print("Loading target assets...", flush=True)
        for symbol in cfg.target_tickers:
            frame, provider = download_symbol(symbol, cfg)
            frames[symbol] = frame
            providers[symbol] = provider
    except Exception as exc:
        if not cfg.allow_synthetic_fallback:
            raise

        print(
            "\nTarget market-data loading failed: "
            f"{type(exc).__name__}: {exc}",
            flush=True,
        )
        print(
            "WARNING: switching to synthetic pipeline-test data.",
            flush=True,
        )
        frames = synthetic_frames(cfg)
        providers = {
            symbol: "synthetic_fallback"
            for symbol in frames
        }
        return (
            frames,
            tuple(
                cfg.market_symbols
                if cfg.include_market_context else ()
            ),
            "synthetic_fallback",
            build_manifest(frames, providers),
        )

    available_market_symbols = []
    if cfg.include_market_context:
        print("Loading market context...", flush=True)
        for symbol in cfg.market_symbols:
            try:
                frame, provider = download_symbol(symbol, cfg)
                frames[symbol] = frame
                providers[symbol] = provider
                available_market_symbols.append(symbol)
            except Exception as exc:
                print(
                    f"  {symbol}: omitted because loading failed: {exc}",
                    flush=True,
                )

    source_name = "real_" + "+".join(
        sorted(set(providers.values()))
    )
    return (
        frames,
        tuple(available_market_symbols),
        source_name,
        build_manifest(frames, providers),
    )


## Feature engineering, forward-only alignment, and dataset construction

In [ ]:
STATISTIC_NAMES = [
    "annualised_mean_return",
    "log_annualised_volatility",
    "mean_absolute_return",
    "downside_annualised_volatility",
    "upside_annualised_volatility",
    "maximum_absolute_return",
    "last_return",
    "cumulative_return",
    "lag1_return_autocorrelation",
    "skewness",
    "excess_kurtosis",
    "volatility_of_volatility",
    "squared_return_trend",
    "log_realised_volatility_5",
    "log_realised_volatility_10",
    "log_realised_volatility_20",
    "log_realised_volatility_60",
    "volatility_ratio_5_20",
    "volatility_ratio_20_60",
    "absolute_return_autocorrelation",
    "squared_return_autocorrelation",
    "large_return_fraction",
    "maximum_drawdown",
    "current_return_streak",
]

OHLCV_FEATURE_NAMES = [
    "mean_log_intraday_range",
    "last_log_intraday_range",
    "log_parkinson_volatility_5",
    "log_parkinson_volatility_20",
    "log_parkinson_volatility_60",
    "mean_absolute_overnight_gap",
    "last_overnight_gap",
    "last_log_relative_volume",
    "mean_log_relative_volume",
    "volume_volatility",
    "missing_volume_fraction",
]


def safe_autocorrelation(values: np.ndarray) -> float:
    if len(values) < 3:
        return 0.0

    left = values[:-1]
    right = values[1:]
    if np.std(left) < EPS or np.std(right) < EPS:
        return 0.0

    result = np.corrcoef(left, right)[0, 1]
    return float(result) if np.isfinite(result) else 0.0


def recent_volatility(
    values: np.ndarray,
    length: int,
    annualisation: float,
) -> float:
    selected = values[-min(length, len(values)):]
    return float(
        np.sqrt(annualisation * np.mean(selected**2))
    )


def volatility_of_volatility(
    values: np.ndarray,
    subwindow: int = 5,
) -> float:
    rolling_mean_square = np.convolve(
        values**2,
        np.ones(subwindow) / subwindow,
        mode="valid",
    )
    return float(
        np.std(np.sqrt(np.maximum(rolling_mean_square, 0.0)))
    )


def maximum_drawdown_from_returns(
    values: np.ndarray,
) -> float:
    cumulative_prices = np.exp(np.cumsum(values))
    running_maximum = np.maximum.accumulate(cumulative_prices)
    return float(
        np.min(cumulative_prices / running_maximum - 1.0)
    )


def current_streak(values: np.ndarray) -> float:
    if len(values) == 0 or values[-1] == 0:
        return 0.0

    final_sign = np.sign(values[-1])
    length = 0
    for value in values[::-1]:
        if np.sign(value) == final_sign:
            length += 1
        else:
            break

    return float(length * final_sign)


def statistical_features(
    past: np.ndarray,
    annualisation: float,
) -> np.ndarray:
    mean_return = float(np.mean(past))
    sample_std = max(float(np.std(past, ddof=1)), EPS)
    annualised_volatility = (
        math.sqrt(annualisation) * sample_std
    )

    downside = np.minimum(past, 0.0)
    upside = np.maximum(past, 0.0)
    standardised = (past - mean_return) / sample_std

    time = np.arange(len(past), dtype=float)
    centred_time = time - time.mean()
    denominator = max(
        float(np.sum(centred_time**2)),
        EPS,
    )
    squared_return_trend = float(
        np.sum(
            centred_time
            * (past**2 - np.mean(past**2))
        )
        / denominator
    )

    vol5 = recent_volatility(past, 5, annualisation)
    vol10 = recent_volatility(past, 10, annualisation)
    vol20 = recent_volatility(past, 20, annualisation)
    vol60 = recent_volatility(past, 60, annualisation)

    large_return_fraction = float(
        np.mean(
            np.abs(past - mean_return)
            > 2.0 * sample_std
        )
    )

    return np.array([
        annualisation * mean_return,
        math.log(annualised_volatility + EPS),
        np.mean(np.abs(past)),
        math.sqrt(
            annualisation * np.mean(downside**2)
        ),
        math.sqrt(
            annualisation * np.mean(upside**2)
        ),
        np.max(np.abs(past)),
        past[-1],
        np.sum(past),
        safe_autocorrelation(past),
        np.mean(standardised**3),
        np.mean(standardised**4) - 3.0,
        volatility_of_volatility(past),
        squared_return_trend,
        math.log(vol5 + EPS),
        math.log(vol10 + EPS),
        math.log(vol20 + EPS),
        math.log(vol60 + EPS),
        vol5 / (vol20 + EPS),
        vol20 / (vol60 + EPS),
        safe_autocorrelation(np.abs(past)),
        safe_autocorrelation(past**2),
        large_return_fraction,
        maximum_drawdown_from_returns(past),
        current_streak(past),
    ], dtype=np.float64)


def ohlcv_features(
    past_frame: pd.DataFrame,
    annualisation: float,
) -> np.ndarray:
    open_price = past_frame["Open"].to_numpy(dtype=float)
    high = past_frame["High"].to_numpy(dtype=float)
    low = past_frame["Low"].to_numpy(dtype=float)
    close = past_frame["Close"].to_numpy(dtype=float)
    volume = past_frame["Volume"].to_numpy(dtype=float)
    volume_missing = past_frame[
        "VolumeMissing"
    ].to_numpy(dtype=float)

    log_range = np.log(
        np.maximum(high, EPS) / np.maximum(low, EPS)
    )
    previous_close = np.concatenate(
        [[close[0]], close[:-1]]
    )
    overnight_gap = np.log(
        np.maximum(open_price, EPS)
        / np.maximum(previous_close, EPS)
    )

    def parkinson(length: int) -> float:
        selected = log_range[-min(length, len(log_range)):]
        variance = (
            np.mean(selected**2)
            / (4.0 * math.log(2.0))
        )
        return math.sqrt(
            annualisation * max(variance, EPS)
        )

    volume_series = pd.Series(volume)
    rolling_volume = volume_series.rolling(
        20,
        min_periods=5,
    ).mean().to_numpy()

    positive_volume = volume[volume > 0]
    fallback_volume = (
        float(np.median(positive_volume))
        if len(positive_volume) else 1.0
    )
    rolling_volume = np.where(
        np.isfinite(rolling_volume)
        & (rolling_volume > 0),
        rolling_volume,
        fallback_volume,
    )
    log_relative_volume = np.log(
        (volume + 1.0) / (rolling_volume + 1.0)
    )

    return np.array([
        np.mean(log_range),
        log_range[-1],
        math.log(parkinson(5) + EPS),
        math.log(parkinson(20) + EPS),
        math.log(parkinson(60) + EPS),
        np.mean(np.abs(overnight_gap)),
        overnight_gap[-1],
        log_relative_volume[-1],
        np.mean(log_relative_volume),
        np.std(log_relative_volume),
        np.mean(volume_missing),
    ], dtype=np.float64)


def market_feature_names(
    symbols: tuple[str, ...],
) -> list[str]:
    names = []
    for symbol in symbols:
        safe_symbol = symbol.lower().replace("^", "")
        if symbol == "^VIX":
            names.extend([
                f"{safe_symbol}_level",
                f"{safe_symbol}_change_5",
                f"{safe_symbol}_mean_20",
                f"{safe_symbol}_volatility_20",
            ])
        else:
            names.extend([
                f"{safe_symbol}_log_volatility_5",
                f"{safe_symbol}_log_volatility_20",
                f"{safe_symbol}_log_volatility_60",
                f"{safe_symbol}_return_5",
                f"{safe_symbol}_return_20",
                f"{safe_symbol}_downside_volatility_20",
            ])
    return names


def align_market_data_forward_only(
    target_dates: pd.DatetimeIndex,
    frames: dict[str, pd.DataFrame],
    symbols: tuple[str, ...],
    forward_fill_limit: int,
) -> dict[str, dict[str, np.ndarray]]:
    aligned = {}

    for symbol in symbols:
        raw_close = frames[symbol]["Close"].reindex(target_dates)
        close = raw_close.ffill(limit=forward_fill_limit)

        # No backward fill is permitted.
        returns = np.log(close).diff()

        aligned[symbol] = {
            "close": close.to_numpy(dtype=float),
            "returns": returns.to_numpy(dtype=float),
            "originally_missing": raw_close.isna().to_numpy(),
        }

    return aligned


def market_features_or_none(
    aligned_market_data: dict[str, dict[str, np.ndarray]],
    end_index: int,
    symbols: tuple[str, ...],
    cfg: Config,
) -> np.ndarray | None:
    values = []

    for symbol in symbols:
        information = aligned_market_data[symbol]
        close = information["close"][
            end_index - cfg.window:end_index
        ]
        returns = information["returns"][
            end_index - cfg.window:end_index
        ]

        if (
            len(close) != cfg.window
            or len(returns) != cfg.window
            or not np.all(np.isfinite(close))
            or not np.all(np.isfinite(returns))
        ):
            return None

        if symbol == "^VIX":
            values.extend([
                close[-1],
                close[-1] / (close[-6] + EPS) - 1.0,
                np.mean(close[-20:]),
                np.std(close[-20:]),
            ])
        else:
            downside = np.minimum(returns[-20:], 0.0)
            values.extend([
                math.log(
                    recent_volatility(
                        returns,
                        5,
                        cfg.annualisation,
                    )
                    + EPS
                ),
                math.log(
                    recent_volatility(
                        returns,
                        20,
                        cfg.annualisation,
                    )
                    + EPS
                ),
                math.log(
                    recent_volatility(
                        returns,
                        60,
                        cfg.annualisation,
                    )
                    + EPS
                ),
                np.sum(returns[-5:]),
                np.sum(returns[-20:]),
                math.sqrt(
                    cfg.annualisation
                    * np.mean(downside**2)
                ),
            ])

    return np.asarray(values, dtype=np.float64)


def cumulative_channel_path(channels: np.ndarray) -> np.ndarray:
    channels = np.asarray(channels, dtype=np.float64)
    if channels.ndim == 1:
        channels = channels[:, None]
    cumulative = np.vstack([
        np.zeros((1, channels.shape[1]), dtype=np.float64),
        np.cumsum(channels, axis=0),
    ])
    time_values = np.linspace(0.0, 1.0, len(channels) + 1)[:, None]
    return np.concatenate([time_values, cumulative], axis=1)


def make_shape_signature_path(past: np.ndarray) -> np.ndarray:
    mean = np.mean(past)
    scale = max(float(np.std(past, ddof=1)), EPS)
    standardised = (past - mean) / scale
    channels = np.column_stack([
        standardised / math.sqrt(len(past)),
        standardised**2 / len(past),
    ])
    return cumulative_channel_path(channels)


def make_amplitude_signature_path(
    past: np.ndarray,
    return_scale: float,
) -> np.ndarray:
    scaled = past / return_scale
    channels = np.column_stack([
        scaled / math.sqrt(len(past)),
        scaled**2 / len(past),
    ])
    return cumulative_channel_path(channels)


def make_ohlc_signature_path(
    past: np.ndarray,
    past_frame: pd.DataFrame,
    return_scale: float,
) -> np.ndarray:
    open_price = past_frame["Open"].to_numpy(dtype=float)
    high = past_frame["High"].to_numpy(dtype=float)
    low = past_frame["Low"].to_numpy(dtype=float)
    close = past_frame["Close"].to_numpy(dtype=float)
    intraday_return = np.log(
        np.maximum(close, EPS) / np.maximum(open_price, EPS)
    )
    # Total close-to-close return minus the same-day intraday return equals
    # the overnight component and does not require a pre-window close.
    overnight_return = past - intraday_return
    range_variance = (
        np.log(np.maximum(high, EPS) / np.maximum(low, EPS)) ** 2
        / (4.0 * math.log(2.0))
    )
    channels = np.column_stack([
        (past / return_scale) / math.sqrt(len(past)),
        (range_variance / return_scale**2) / len(past),
        (overnight_return / return_scale) ** 2 / len(past),
    ])
    return cumulative_channel_path(channels)


def make_joint_market_signature_path(
    asset_returns: np.ndarray,
    spy_returns: np.ndarray,
    vix_changes: np.ndarray,
    return_scale: float,
    vix_change_scale: float,
) -> np.ndarray:
    length = len(asset_returns)
    channels = np.column_stack([
        (asset_returns / return_scale) / math.sqrt(length),
        (spy_returns / return_scale) / math.sqrt(length),
        (vix_changes / vix_change_scale) / math.sqrt(length),
    ])
    return cumulative_channel_path(channels)


def make_lead_lag_signature_path(
    past: np.ndarray,
    return_scale: float,
) -> np.ndarray:
    stream = np.concatenate([
        [0.0],
        np.cumsum(past / return_scale) / math.sqrt(len(past)),
    ])
    path = np.empty((2 * len(past) + 1, 2), dtype=np.float64)
    path[0] = (stream[0], stream[0])
    path[1::2, 0] = stream[1:]
    path[1::2, 1] = stream[:-1]
    path[2::2, 0] = stream[1:]
    path[2::2, 1] = stream[1:]
    return path


def signature_block_specs(cfg: Config) -> tuple[SignatureBlockSpec, ...]:
    if not cfg.include_signature_features:
        return ()
    specs = []
    longest_window = max(cfg.signature_windows)
    for window in cfg.signature_windows:
        shape_depth = (
            cfg.signature_depth
            if window == longest_window
            else cfg.short_signature_depth
        )
        if cfg.include_shape_signatures:
            specs.append(
                SignatureBlockSpec("shape", window, 3, shape_depth)
            )
        if cfg.include_amplitude_signatures:
            specs.append(SignatureBlockSpec(
                "amplitude", window, 3, cfg.amplitude_signature_depth
            ))
    for window in cfg.secondary_signature_windows:
        if cfg.include_ohlc_signatures:
            specs.append(SignatureBlockSpec(
                "ohlc", window, 4, cfg.ohlc_signature_depth
            ))
        if cfg.include_joint_market_signatures:
            specs.append(SignatureBlockSpec(
                "joint_market", window, 4, cfg.joint_market_signature_depth
            ))
        if cfg.include_lead_lag_signatures:
            specs.append(SignatureBlockSpec(
                "lead_lag", window, 2, cfg.lead_lag_signature_depth
            ))
    return tuple(specs)


def all_signature_feature_names(cfg: Config) -> list[str]:
    names = []
    for spec in signature_block_specs(cfg):
        names.extend(signature_feature_names(
            spec.prefix,
            spec.path_dimension,
            spec.depth,
        ))
    return names


def signature_features_or_none(
    past: np.ndarray,
    past_frame: pd.DataFrame,
    aligned_market_data: dict[str, dict[str, np.ndarray]],
    end_index: int,
    cfg: Config,
) -> np.ndarray | None:
    pieces = []
    spy_symbol, vix_symbol = cfg.joint_signature_symbols
    for spec in signature_block_specs(cfg):
        selected_returns = past[-spec.window:]
        if spec.family == "shape":
            path = make_shape_signature_path(selected_returns)
        elif spec.family == "amplitude":
            path = make_amplitude_signature_path(
                selected_returns, cfg.signature_return_scale
            )
        elif spec.family == "ohlc":
            path = make_ohlc_signature_path(
                selected_returns,
                past_frame.iloc[-spec.window:],
                cfg.signature_return_scale,
            )
        elif spec.family == "joint_market":
            spy_returns = aligned_market_data[spy_symbol]["returns"][
                end_index - spec.window:end_index
            ]
            vix_changes = aligned_market_data[vix_symbol]["returns"][
                end_index - spec.window:end_index
            ]
            if (
                len(spy_returns) != spec.window
                or len(vix_changes) != spec.window
                or not np.all(np.isfinite(spy_returns))
                or not np.all(np.isfinite(vix_changes))
            ):
                return None
            path = make_joint_market_signature_path(
                selected_returns,
                spy_returns,
                vix_changes,
                cfg.signature_return_scale,
                cfg.signature_vix_change_scale,
            )
        elif spec.family == "lead_lag":
            path = make_lead_lag_signature_path(
                selected_returns, cfg.signature_return_scale
            )
        else:
            raise RuntimeError(f"Unknown signature family: {spec.family}")

        values = truncated_signature(path, spec.depth)
        if not np.all(np.isfinite(values)):
            return None
        pieces.append(values)
    return (
        np.concatenate(pieces)
        if pieces
        else np.empty(0, dtype=np.float64)
    )


def realised_volatility(
    values: np.ndarray,
    annualisation: float,
) -> float:
    return float(
        np.sqrt(annualisation * np.mean(values**2))
    )


def ewma_volatility(
    values: np.ndarray,
    annualisation: float,
    decay: float,
) -> float:
    variance = max(float(np.var(values, ddof=1)), EPS)

    for value in values:
        variance = (
            decay * variance
            + (1.0 - decay) * float(value**2)
        )

    return math.sqrt(
        annualisation * max(variance, EPS)
    )


def feature_groups_for(
    feature_names: list[str],
) -> dict[str, tuple[str, ...]]:
    """Return overlapping, named masks for group and exact-feature ablations."""
    names = set(feature_names)

    def present(candidates: list[str] | tuple[str, ...]) -> tuple[str, ...]:
        return tuple(name for name in candidates if name in names)

    signature = tuple(name for name in feature_names if name.startswith("signature_"))
    signature_shape = tuple(
        name for name in signature if name.startswith("signature_shape_")
    )
    signature_amplitude = tuple(
        name for name in signature if name.startswith("signature_amplitude_")
    )
    signature_ohlc = tuple(
        name for name in signature if name.startswith("signature_ohlc_")
    )
    signature_joint_market = tuple(
        name for name in signature
        if name.startswith("signature_joint_market_")
    )
    signature_lead_lag = tuple(
        name for name in signature if name.startswith("signature_lead_lag_")
    )
    scalar_market = tuple(
        name for name in feature_names
        if name.startswith(("spy_", "qqq_", "vix_"))
    )
    market = tuple(dict.fromkeys(scalar_market + signature_joint_market))
    scalar_ohlcv = present(OHLCV_FEATURE_NAMES)
    ohlcv = tuple(dict.fromkeys(scalar_ohlcv + signature_ohlc))

    return {
        "signature": signature,
        "signature_shape": signature_shape,
        "signature_amplitude": signature_amplitude,
        "signature_ohlc": signature_ohlc,
        "signature_joint_market": signature_joint_market,
        "signature_lead_lag": signature_lead_lag,
        "signature_w10": tuple(name for name in signature if "_w10_" in name),
        "signature_w20": tuple(name for name in signature if "_w20_" in name),
        "signature_w60": tuple(name for name in signature if "_w60_" in name),
        "signature_l1": tuple(name for name in signature if "_L1_" in name),
        "signature_l2": tuple(name for name in signature if "_L2_" in name),
        "signature_l3": tuple(name for name in signature if "_L3_" in name),
        "return_statistics": present(STATISTIC_NAMES),
        "return_location_scale": present((
            "annualised_mean_return", "log_annualised_volatility",
            "mean_absolute_return", "downside_annualised_volatility",
            "upside_annualised_volatility",
        )),
        "return_tail_shape": present((
            "maximum_absolute_return", "skewness", "excess_kurtosis",
            "large_return_fraction", "maximum_drawdown",
        )),
        "return_dynamics": present((
            "last_return", "cumulative_return", "lag1_return_autocorrelation",
            "volatility_of_volatility", "squared_return_trend",
            "absolute_return_autocorrelation", "squared_return_autocorrelation",
            "current_return_streak",
        )),
        "realised_volatility_horizons": present((
            "log_realised_volatility_5", "log_realised_volatility_10",
            "log_realised_volatility_20", "log_realised_volatility_60",
            "volatility_ratio_5_20", "volatility_ratio_20_60",
        )),
        "ohlcv": ohlcv,
        "ohlcv_range": present((
            "mean_log_intraday_range", "last_log_intraday_range",
            "log_parkinson_volatility_5", "log_parkinson_volatility_20",
            "log_parkinson_volatility_60",
        )),
        "ohlcv_overnight": present((
            "mean_absolute_overnight_gap", "last_overnight_gap",
        )),
        "ohlcv_volume": present((
            "last_log_relative_volume", "mean_log_relative_volume",
            "volume_volatility", "missing_volume_fraction",
        )),
        "market": market,
        "market_spy": tuple(name for name in scalar_market if name.startswith("spy_")) + signature_joint_market,
        "market_qqq": tuple(name for name in scalar_market if name.startswith("qqq_")),
        "market_vix": tuple(name for name in scalar_market if name.startswith("vix_")) + signature_joint_market,
    }


def build_market_dataset(cfg: Config) -> MarketDataset:
    (
        frames,
        available_market_symbols,
        data_source,
        data_manifest,
    ) = load_all_frames(cfg)

    data_manifest = data_manifest.copy()
    data_manifest["role"] = data_manifest["symbol"].map(
        lambda symbol: (
            "training_ticker" if symbol in cfg.training_tickers
            else "unseen_test_ticker" if symbol in cfg.unseen_test_tickers
            else "market_context"
        )
    )
    data_manifest["sector"] = data_manifest["symbol"].map(TICKER_SECTORS).fillna(
        "market_context"
    )

    if (
        cfg.experiment_mode == "real_experiment"
        and data_source.startswith("synthetic")
    ):
        raise RuntimeError(
            "A final real experiment may not use synthetic data."
        )

    feature_rows = []
    target_rows = []
    metadata_rows = []
    skipped_rows = []

    required_joint_symbols = set(cfg.joint_signature_symbols)
    if (
        cfg.include_signature_features
        and cfg.include_joint_market_signatures
        and not required_joint_symbols <= set(available_market_symbols)
    ):
        raise RuntimeError(
            "Joint market signatures require all configured symbols; missing: "
            f"{sorted(required_joint_symbols - set(available_market_symbols))}"
        )

    feature_names = (
        all_signature_feature_names(cfg)
        + STATISTIC_NAMES
        + OHLCV_FEATURE_NAMES
        + market_feature_names(available_market_symbols)
    )

    print("\nBuilding leakage-controlled samples...")

    for ticker_id, ticker in enumerate(cfg.target_tickers):
        frame = frames[ticker].copy()

        close_returns = np.log(
            frame["Close"]
        ).diff().dropna()
        dates = pd.DatetimeIndex(close_returns.index)
        returns = close_returns.to_numpy(dtype=np.float64)
        aligned_asset_frame = frame.reindex(dates)

        aligned_markets = align_market_data_forward_only(
            dates,
            frames,
            available_market_symbols,
            cfg.market_forward_fill_limit,
        )

        accepted = 0
        skipped = 0

        for index in range(
            cfg.window,
            len(returns) - cfg.horizon + 1,
        ):
            past = returns[index - cfg.window:index]
            future = returns[index:index + cfg.horizon]
            past_frame = aligned_asset_frame.iloc[
                index - cfg.window:index
            ]

            if past_frame[REQUIRED_PRICE_COLUMNS].isna().any().any():
                skipped_rows.append({
                    "ticker": ticker,
                    "origin_date": dates[index - 1],
                    "reason": "missing_target_ohlc",
                })
                skipped += 1
                continue

            market_values = market_features_or_none(
                aligned_markets,
                index,
                available_market_symbols,
                cfg,
            )
            if market_values is None:
                skipped_rows.append({
                    "ticker": ticker,
                    "origin_date": dates[index - 1],
                    "reason": "insufficient_forward_only_market_context",
                })
                skipped += 1
                continue

            signature_values = signature_features_or_none(
                past,
                past_frame,
                aligned_markets,
                index,
                cfg,
            )
            if signature_values is None:
                skipped_rows.append({
                    "ticker": ticker,
                    "origin_date": dates[index - 1],
                    "reason": "insufficient_signature_context",
                })
                skipped += 1
                continue

            feature = np.concatenate([
                signature_values,
                statistical_features(
                    past,
                    cfg.annualisation,
                ),
                ohlcv_features(
                    past_frame,
                    cfg.annualisation,
                ),
                market_values,
            ])

            actual_volatility = realised_volatility(
                future,
                cfg.annualisation,
            )
            rolling_baseline = recent_volatility(
                past, 20, cfg.annualisation
            )

            feature_rows.append(feature)
            target_rows.append(
                math.log(actual_volatility + EPS)
            )
            metadata_rows.append({
                "ticker_id": ticker_id,
                "ticker": ticker,
                "ticker_role": (
                    "training" if ticker in cfg.training_tickers
                    else "unseen_test"
                ),
                "origin_date": dates[index - 1],
                "target_start_date": dates[index],
                "target_end_date": dates[
                    index + cfg.horizon - 1
                ],
                "actual_vol": actual_volatility,
                "rolling_vol_baseline": rolling_baseline,
                "future_to_recent_vol_ratio": (
                    actual_volatility / max(rolling_baseline, EPS)
                ),
                "sector": TICKER_SECTORS[ticker],
                "ewma_vol_baseline": ewma_volatility(
                    past,
                    cfg.annualisation,
                    cfg.ewma_lambda,
                ),
                "data_source": data_source,
            })
            accepted += 1

        print(
            f"  {ticker}: accepted={accepted:,}, skipped={skipped:,}"
        )

    if not feature_rows:
        raise RuntimeError("No samples survived the leakage controls.")

    unsorted_metadata = pd.DataFrame(metadata_rows)
    ordering = (
        unsorted_metadata.assign(
            _position=np.arange(len(unsorted_metadata))
        )
        .sort_values(["origin_date", "ticker"])["_position"]
        .to_numpy(dtype=int)
    )

    metadata = (
        unsorted_metadata
        .iloc[ordering]
        .reset_index(drop=True)
    )
    features = np.asarray(
        feature_rows,
        dtype=np.float32,
    )[ordering]
    targets = np.asarray(
        target_rows,
        dtype=np.float32,
    )[ordering]

    if len(feature_names) != len(set(feature_names)):
        raise RuntimeError("Canonical feature names are not unique.")
    if features.shape[1] != len(feature_names):
        raise RuntimeError(
            f"Feature mismatch: values={features.shape[1]}, "
            f"names={len(feature_names)}"
        )

    if not np.all(np.isfinite(features)):
        raise ValueError("Non-finite feature values remain.")

    if not np.all(np.isfinite(targets)):
        raise ValueError("Non-finite target values remain.")

    skipped_samples = pd.DataFrame(
        skipped_rows,
        columns=["ticker", "origin_date", "reason"],
    )

    print(
        f"Dataset complete: {len(metadata):,} samples, "
        f"{features.shape[1]} features, source={data_source}"
    )

    return MarketDataset(
        features=features,
        targets_log_vol=targets,
        metadata=metadata,
        feature_names=feature_names,
        data_source=data_source,
        available_market_symbols=available_market_symbols,
        data_manifest=data_manifest,
        skipped_samples=skipped_samples,
    )


## Purged outer splits, inner development folds, and transferable regimes

The outer protocol reserves training, validation, calibration, and a final 15% time section. Feature or objective selection uses three expanding folds entirely inside the outer training section. The untouched outer validation selects checkpoints, calibration alone tunes intervals/temperature, and the final test is opened once.

Regimes are Contracting, Stable, and Expanding relative to each row's past-only 20-session volatility. Their symmetric boundaries are fitted from training targets only; held-out tickers receive pooled training boundaries.


In [ ]:
def cutoff_from_fraction(
    unique_dates: np.ndarray,
    fraction: float,
) -> pd.Timestamp:
    position = int(len(unique_dates) * fraction) - 1
    position = max(0, min(position, len(unique_dates) - 1))
    return pd.Timestamp(unique_dates[position])


def resolve_active_feature_set(
    dataset: MarketDataset,
    cfg: Config,
) -> tuple[np.ndarray, list[str], list[str]]:
    feature_names = dataset.feature_names
    groups = feature_groups_for(feature_names)
    unknown_groups = sorted(set(cfg.disabled_feature_groups) - set(groups))
    if unknown_groups:
        raise ValueError(
            f"Unknown disabled feature groups: {unknown_groups}. "
            f"Valid groups: {sorted(groups)}"
        )

    disabled = set(cfg.disabled_feature_names)
    unknown_features = sorted(disabled - set(feature_names))
    if unknown_features:
        raise ValueError(
            f"Unknown disabled features: {unknown_features}. "
            "Use the exact names printed in the feature manifest."
        )

    if not cfg.include_signature_features:
        disabled.update(groups["signature"])
    if not cfg.include_shape_signatures:
        disabled.update(groups["signature_shape"])
    if not cfg.include_amplitude_signatures:
        disabled.update(groups["signature_amplitude"])
    if not cfg.include_ohlc_signatures:
        disabled.update(groups["signature_ohlc"])
    if not cfg.include_joint_market_signatures:
        disabled.update(groups["signature_joint_market"])
    if not cfg.include_lead_lag_signatures:
        disabled.update(groups["signature_lead_lag"])
    if not cfg.include_return_statistics:
        disabled.update(groups["return_statistics"])
    if not cfg.include_ohlcv_features:
        disabled.update(groups["ohlcv"])
    if not cfg.include_market_features:
        disabled.update(groups["market"])

    for group_name in cfg.disabled_feature_groups:
        disabled.update(groups[group_name])

    active_indices = np.array(
        [index for index, name in enumerate(feature_names) if name not in disabled],
        dtype=int,
    )
    active_names = [feature_names[index] for index in active_indices]
    dropped_names = [name for name in feature_names if name in disabled]

    if len(active_indices) == 0:
        raise ValueError("Every canonical feature was disabled.")

    print("\nModel-input feature manifest:")
    print(f"  Canonical engineered features: {len(feature_names)}")
    print(f"  Active engineered features:    {len(active_names)}")
    print(f"  Dropped engineered features:   {len(dropped_names)}")
    print(f"  Seen-ticker identity information enabled: {cfg.include_ticker_identity}")
    print("  Valid feature groups:", ", ".join(sorted(groups)))
    print("  Active feature names:")
    print("   ", ", ".join(active_names))
    if dropped_names:
        print("  Dropped feature names:")
        print("   ", ", ".join(dropped_names))

    return active_indices, active_names, dropped_names


def regime_score_values(
    dataset: MarketDataset,
    cfg: Config,
) -> np.ndarray:
    if cfg.regime_target_mode == "absolute_volatility":
        return dataset.targets_log_vol.astype(float)
    recent = dataset.metadata["rolling_vol_baseline"].to_numpy(dtype=float)
    return (
        dataset.targets_log_vol.astype(float)
        - np.log(np.maximum(recent, EPS))
    )


def regime_boundaries(values: np.ndarray, relative: bool) -> np.ndarray:
    if relative:
        half_width = float(np.quantile(np.abs(values), 1.0 / 3.0))
        return np.array([-half_width, half_width], dtype=np.float32)
    return np.quantile(values, [1 / 3, 2 / 3]).astype(np.float32)


def build_ticker_thresholds_and_labels(
    dataset: MarketDataset,
    train_indices: np.ndarray,
    cfg: Config,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    metadata = dataset.metadata
    scores = regime_score_values(dataset, cfg)
    relative = cfg.regime_target_mode == "relative_to_recent_volatility"
    pooled_thresholds = regime_boundaries(scores[train_indices], relative)

    thresholds = np.tile(
        pooled_thresholds[None, :],
        (len(cfg.target_tickers), 1),
    ).astype(np.float32)
    sources = np.full(
        len(cfg.target_tickers),
        f"pooled_training_tickers::{cfg.regime_target_mode}",
        dtype=object,
    )
    labels = np.zeros(len(metadata), dtype=np.int64)

    for ticker_id, ticker in enumerate(cfg.target_tickers):
        ticker_train_indices = train_indices[
            metadata.iloc[train_indices]["ticker_id"].to_numpy(dtype=int)
            == ticker_id
        ]

        if ticker in cfg.training_tickers and len(ticker_train_indices) < 30:
            raise ValueError(f"Too few training samples for {ticker}.")

        if (
            cfg.regime_threshold_mode == "per_training_ticker"
            and ticker in cfg.training_tickers
        ):
            thresholds[ticker_id] = regime_boundaries(
                scores[ticker_train_indices], relative
            )
            sources[ticker_id] = (
                f"{ticker}_training_only::{cfg.regime_target_mode}"
            )

        ticker_indices = np.flatnonzero(
            metadata["ticker_id"].to_numpy(dtype=int) == ticker_id
        )
        labels[ticker_indices] = np.digitize(
            scores[ticker_indices],
            thresholds[ticker_id],
        )

    return thresholds, labels, sources


def build_training_regime_statistics(
    metadata: pd.DataFrame,
    labels: np.ndarray,
    train_indices: np.ndarray,
    cfg: Config,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    counts = np.bincount(
        labels[train_indices],
        minlength=cfg.regimes,
    ).astype(float)
    proportions = counts / counts.sum()

    class_weights = counts.sum() / (
        cfg.regimes * np.maximum(counts, 1.0)
    )
    class_weights[1] *= cfg.medium_class_multiplier
    class_weights /= class_weights.mean()

    training_set = set(train_indices.tolist())
    pooled_transition = np.ones((cfg.regimes, cfg.regimes), dtype=float)
    pooled_initial = np.ones(cfg.regimes, dtype=float)
    ordered_training_indices: dict[int, np.ndarray] = {}

    for ticker_id, ticker in enumerate(cfg.target_tickers):
        if ticker not in cfg.training_tickers:
            continue
        ordered = (
            metadata[metadata["ticker_id"] == ticker_id]
            .sort_values("origin_date")
            .index.to_numpy(dtype=int)
        )
        ordered = np.array(
            [index for index in ordered if index in training_set],
            dtype=int,
        )
        ordered_training_indices[ticker_id] = ordered
        if len(ordered) == 0:
            continue
        pooled_initial[labels[ordered[0]]] += 1.0
        for previous, current in zip(labels[ordered[:-1]], labels[ordered[1:]]):
            pooled_transition[previous, current] += 1.0

    pooled_transition /= pooled_transition.sum(axis=1, keepdims=True)
    pooled_initial /= pooled_initial.sum()

    transitions = np.tile(
        pooled_transition[None, :, :],
        (len(cfg.target_tickers), 1, 1),
    )
    initial = np.tile(
        pooled_initial[None, :],
        (len(cfg.target_tickers), 1),
    )

    if cfg.ticker_specific_regime_smoothing:
        for ticker_id, ordered in ordered_training_indices.items():
            local_transition = np.ones((cfg.regimes, cfg.regimes), dtype=float)
            local_initial = np.ones(cfg.regimes, dtype=float)
            if len(ordered):
                local_initial[labels[ordered[0]]] += 1.0
                for previous, current in zip(
                    labels[ordered[:-1]], labels[ordered[1:]]
                ):
                    local_transition[previous, current] += 1.0
            local_transition /= local_transition.sum(axis=1, keepdims=True)
            local_initial /= local_initial.sum()
            transitions[ticker_id] = local_transition
            initial[ticker_id] = local_initial

    return (
        class_weights.astype(np.float32),
        proportions.astype(np.float32),
        transitions.astype(np.float32),
        initial.astype(np.float32),
    )


def assert_split_integrity(
    dataset: MarketDataset,
    train_indices: np.ndarray,
    validation_indices: np.ndarray,
    calibration_indices: np.ndarray,
    seen_ticker_unseen_day_indices: np.ndarray,
    unseen_ticker_unseen_day_indices: np.ndarray,
    train_cutoff: pd.Timestamp,
    validation_cutoff: pd.Timestamp,
    calibration_cutoff: pd.Timestamp,
    cfg: Config,
) -> pd.DataFrame:
    metadata = dataset.metadata
    sections = {
        "train": train_indices,
        "validation": validation_indices,
        "calibration": calibration_indices,
        "seen_ticker_unseen_day": seen_ticker_unseen_day_indices,
        "unseen_ticker_unseen_day": unseen_ticker_unseen_day_indices,
    }
    all_indices = np.concatenate(list(sections.values()))
    if len(np.unique(all_indices)) != len(all_indices):
        raise AssertionError("Split index sets overlap.")

    checks: list[dict[str, object]] = []

    def check(description: str, passed: bool, observed: object, boundary: object) -> None:
        checks.append({
            "check": description,
            "passed": bool(passed),
            "observed": observed,
            "boundary": boundary,
        })
        if not passed:
            raise AssertionError(description)

    fit_indices = np.concatenate([
        train_indices, validation_indices, calibration_indices,
    ])
    fit_tickers = set(metadata.iloc[fit_indices]["ticker"])
    check(
        "held-out tickers absent from train, validation, and calibration",
        fit_tickers.isdisjoint(cfg.unseen_test_tickers),
        sorted(fit_tickers & set(cfg.unseen_test_tickers)),
        "empty",
    )
    check(
        "training targets end by training cutoff",
        pd.to_datetime(metadata.iloc[train_indices]["target_end_date"]).max()
        <= train_cutoff,
        pd.to_datetime(metadata.iloc[train_indices]["target_end_date"]).max(),
        train_cutoff,
    )
    check(
        "validation origins begin after training cutoff",
        pd.to_datetime(metadata.iloc[validation_indices]["origin_date"]).min()
        > train_cutoff,
        pd.to_datetime(metadata.iloc[validation_indices]["origin_date"]).min(),
        train_cutoff,
    )
    check(
        "validation targets end by validation cutoff",
        pd.to_datetime(metadata.iloc[validation_indices]["target_end_date"]).max()
        <= validation_cutoff,
        pd.to_datetime(metadata.iloc[validation_indices]["target_end_date"]).max(),
        validation_cutoff,
    )
    check(
        "calibration origins begin after validation cutoff",
        pd.to_datetime(metadata.iloc[calibration_indices]["origin_date"]).min()
        > validation_cutoff,
        pd.to_datetime(metadata.iloc[calibration_indices]["origin_date"]).min(),
        validation_cutoff,
    )
    check(
        "calibration targets end by calibration cutoff",
        pd.to_datetime(metadata.iloc[calibration_indices]["target_end_date"]).max()
        <= calibration_cutoff,
        pd.to_datetime(metadata.iloc[calibration_indices]["target_end_date"]).max(),
        calibration_cutoff,
    )

    for cohort_name, indices, allowed_tickers in (
        (
            "seen-ticker unseen-day",
            seen_ticker_unseen_day_indices,
            set(cfg.training_tickers),
        ),
        (
            "unseen-ticker unseen-day",
            unseen_ticker_unseen_day_indices,
            set(cfg.unseen_test_tickers),
        ),
    ):
        cohort_tickers = set(metadata.iloc[indices]["ticker"])
        cohort_origin_min = pd.to_datetime(
            metadata.iloc[indices]["origin_date"]
        ).min()
        check(
            f"{cohort_name} origins begin after calibration cutoff",
            cohort_origin_min > calibration_cutoff,
            cohort_origin_min,
            calibration_cutoff,
        )
        check(
            f"{cohort_name} contains every configured ticker in its role",
            cohort_tickers == allowed_tickers,
            sorted(cohort_tickers),
            sorted(allowed_tickers),
        )

    seen_dates = set(pd.to_datetime(
        metadata.iloc[seen_ticker_unseen_day_indices]["origin_date"]
    ))
    unseen_dates = set(pd.to_datetime(
        metadata.iloc[unseen_ticker_unseen_day_indices]["origin_date"]
    ))
    check(
        "seen and unseen ticker cohorts use the identical origin-date calendar",
        seen_dates == unseen_dates,
        len(seen_dates),
        len(unseen_dates),
    )

    return pd.DataFrame(checks)


def prepare_split(
    dataset: MarketDataset,
    cfg: Config,
) -> PreparedSplit:
    metadata = dataset.metadata
    training_ticker_mask = metadata["ticker"].isin(cfg.training_tickers)
    unseen_ticker_mask = metadata["ticker"].isin(cfg.unseen_test_tickers)
    training_dates = np.array(sorted(pd.to_datetime(
        metadata.loc[training_ticker_mask, "origin_date"]
    ).unique()))

    train_cutoff = cutoff_from_fraction(training_dates, cfg.train_fraction)
    validation_cutoff = cutoff_from_fraction(
        training_dates, cfg.train_fraction + cfg.validation_fraction
    )
    calibration_cutoff = cutoff_from_fraction(
        training_dates,
        cfg.train_fraction + cfg.validation_fraction + cfg.calibration_fraction,
    )

    origin = pd.to_datetime(metadata["origin_date"])
    target_end = pd.to_datetime(metadata["target_end_date"])

    train_indices = np.flatnonzero(
        (training_ticker_mask & (target_end <= train_cutoff)).to_numpy()
    )
    validation_indices = np.flatnonzero((
        training_ticker_mask
        & (origin > train_cutoff)
        & (target_end <= validation_cutoff)
    ).to_numpy())
    calibration_indices = np.flatnonzero((
        training_ticker_mask
        & (origin > validation_cutoff)
        & (target_end <= calibration_cutoff)
    ).to_numpy())
    post_calibration_date_sets = []
    for ticker in cfg.target_tickers:
        ticker_dates = set(pd.to_datetime(metadata.loc[
            (metadata["ticker"] == ticker) & (origin > calibration_cutoff),
            "origin_date",
        ]))
        post_calibration_date_sets.append(ticker_dates)
    common_test_dates = set.intersection(*post_calibration_date_sets)
    if len(common_test_dates) < cfg.minimum_test_origins_per_ticker:
        raise ValueError(
            "Too few common post-calibration origins across every configured "
            f"ticker: {len(common_test_dates)} < "
            f"{cfg.minimum_test_origins_per_ticker}. Check cache coverage."
        )
    common_test_date_mask = origin.isin(common_test_dates)
    seen_ticker_unseen_day_indices = np.flatnonzero((
        training_ticker_mask & common_test_date_mask
    ).to_numpy())
    unseen_ticker_unseen_day_indices = np.flatnonzero((
        unseen_ticker_mask & common_test_date_mask
    ).to_numpy())
    test_indices = np.sort(np.concatenate([
        seen_ticker_unseen_day_indices,
        unseen_ticker_unseen_day_indices,
    ]))

    if min(
        len(train_indices),
        len(validation_indices),
        len(calibration_indices),
        len(seen_ticker_unseen_day_indices),
        len(unseen_ticker_unseen_day_indices),
    ) == 0:
        raise ValueError("At least one fitting section or test cohort is empty.")

    leakage_audit = assert_split_integrity(
        dataset,
        train_indices,
        validation_indices,
        calibration_indices,
        seen_ticker_unseen_day_indices,
        unseen_ticker_unseen_day_indices,
        train_cutoff,
        validation_cutoff,
        calibration_cutoff,
        cfg,
    )

    active_indices, active_names, dropped_names = resolve_active_feature_set(
        dataset, cfg
    )
    standardiser = RobustStandardiser.fit(
        dataset.features[train_indices],
        cfg.winsor_lower_quantile,
        cfg.winsor_upper_quantile,
        cfg.standardised_clip,
    )
    scaled_features = standardiser.transform(dataset.features)
    active_mask = np.zeros(len(dataset.feature_names), dtype=bool)
    active_mask[active_indices] = True
    # Zero masking keeps canonical width, fan-in, parameter count, and seed
    # behaviour fixed across every engineered-feature ablation.
    scaled_features[:, ~active_mask] = 0.0

    ticker_identity = np.zeros(
        (len(metadata), len(cfg.training_tickers)),
        dtype=np.float32,
    )
    if cfg.include_ticker_identity:
        ticker_values = metadata["ticker"].to_numpy()
        for identity_index, ticker in enumerate(cfg.training_tickers):
            ticker_identity[ticker_values == ticker, identity_index] = 1.0

    identity_names = [
        f"ticker_identity_{ticker}" for ticker in cfg.training_tickers
    ]
    context = np.concatenate(
        [scaled_features, ticker_identity],
        axis=1,
    ).astype(np.float32)
    context_feature_names = list(dataset.feature_names) + identity_names

    ticker_thresholds, labels, threshold_sources = (
        build_ticker_thresholds_and_labels(dataset, train_indices, cfg)
    )
    (
        class_weights,
        train_proportions,
        transition_matrices,
        initial_probabilities,
    ) = build_training_regime_statistics(
        metadata, labels, train_indices, cfg
    )

    metadata.loc[:, "split"] = "not_used_for_fitting_or_final_test"
    metadata.loc[train_indices, "split"] = "train"
    metadata.loc[validation_indices, "split"] = "validation"
    metadata.loc[calibration_indices, "split"] = "calibration"
    metadata.loc[test_indices, "split"] = "test"
    metadata.loc[:, "evaluation_cohort"] = "not_evaluated"
    metadata.loc[
        seen_ticker_unseen_day_indices, "evaluation_cohort"
    ] = "seen_ticker_unseen_day"
    metadata.loc[
        unseen_ticker_unseen_day_indices, "evaluation_cohort"
    ] = "unseen_ticker_unseen_day"
    metadata.loc[:, "regime"] = labels
    metadata.loc[:, "regime_name"] = REGIME_NAMES[labels]
    metadata.loc[:, "regime_threshold_source"] = threshold_sources[
        metadata["ticker_id"].to_numpy(dtype=int)
    ]

    print("\nStrict fitting sections and final generalisation cohorts:")
    print(f"  Train:                     {len(train_indices):,}")
    print(f"  Validation:                {len(validation_indices):,}")
    print(f"  Calibration:               {len(calibration_indices):,}")
    print(f"  Seen ticker / unseen day:  {len(seen_ticker_unseen_day_indices):,}")
    print(f"  Unseen ticker / unseen day:{len(unseen_ticker_unseen_day_indices):,}")
    print(f"  Train cutoff:       {train_cutoff.date()}")
    print(f"  Validation cutoff:  {validation_cutoff.date()}")
    print(f"  Calibration cutoff: {calibration_cutoff.date()}")
    print(f"  Common final-test origin dates: {len(common_test_dates):,}")

    boundary_units = (
        "future/recent volatility ratio"
        if cfg.regime_target_mode == "relative_to_recent_volatility"
        else "annualised volatility"
    )
    print(f"\nTraining-only regime boundaries ({boundary_units}):")
    for ticker_id, ticker in enumerate(cfg.target_tickers):
        print(
            f"  {ticker}: {math.exp(ticker_thresholds[ticker_id, 0]):.3f}, "
            f"{math.exp(ticker_thresholds[ticker_id, 1]):.3f} "
            f"[{threshold_sources[ticker_id]}]"
        )

    print("\nLeakage and cohort audit:")
    print(leakage_audit.to_string(index=False))

    return PreparedSplit(
        train_indices=train_indices,
        validation_indices=validation_indices,
        calibration_indices=calibration_indices,
        test_indices=test_indices,
        seen_ticker_unseen_day_indices=seen_ticker_unseen_day_indices,
        unseen_ticker_unseen_day_indices=unseen_ticker_unseen_day_indices,
        context=context,
        regime_labels=labels,
        ticker_thresholds=ticker_thresholds,
        class_weights=class_weights,
        train_regime_proportions=train_proportions,
        transition_matrices=transition_matrices,
        initial_regime_probabilities=initial_probabilities,
        standardiser=standardiser,
        active_feature_indices=active_indices,
        active_feature_names=active_names,
        dropped_feature_names=dropped_names,
        context_feature_names=context_feature_names,
        train_cutoff=train_cutoff,
        validation_cutoff=validation_cutoff,
        calibration_cutoff=calibration_cutoff,
        leakage_audit=leakage_audit,
    )


## Single-pass conditional flow mixture

In [ ]:
def stable_log_cosh(value: torch.Tensor) -> torch.Tensor:
    return (
        torch.logaddexp(value, -value)
        - math.log(2.0)
    )


@dataclass
class ForwardBundle:
    logits: torch.Tensor
    probabilities: torch.Tensor
    locations: torch.Tensor
    scales: torch.Tensor
    skews: torch.Tensor
    tails: torch.Tensor
    expert_medians: torch.Tensor
    mixture_median_approximation: torch.Tensor
    quantile_lower: torch.Tensor
    quantile_median: torch.Tensor
    quantile_upper: torch.Tensor
    component_log_probabilities: torch.Tensor | None
    mixture_log_probability: torch.Tensor | None


class RegimeFlowMixture(nn.Module):
    def __init__(
        self,
        context_dimension: int,
        cfg: Config,
    ):
        super().__init__()
        self.cfg = cfg

        self.encoder = nn.Sequential(
            nn.Linear(
                context_dimension,
                cfg.hidden_size,
            ),
            nn.LayerNorm(cfg.hidden_size),
            nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(
                cfg.hidden_size,
                cfg.hidden_size,
            ),
            nn.SiLU(),
        )

        self.gate_head = nn.Linear(
            cfg.hidden_size,
            cfg.regimes,
        )
        self.quantile_head = nn.Linear(
            cfg.hidden_size,
            3,
        )
        self.expert_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(
                    cfg.hidden_size,
                    cfg.hidden_size,
                ),
                nn.SiLU(),
                nn.Linear(cfg.hidden_size, 4),
            )
            for _ in range(cfg.regimes)
        ])

        for head in self.expert_heads:
            final_layer = head[-1]
            nn.init.zeros_(final_layer.weight)
            with torch.no_grad():
                final_layer.bias[:] = torch.tensor([
                    -1.8,
                    0.0,
                    0.0,
                    0.0,
                ])

    def forward_bundle(
        self,
        context: torch.Tensor,
        target: torch.Tensor | None = None,
    ) -> ForwardBundle:
        # Exactly one stochastic encoder pass per batch.
        hidden = self.encoder(context)

        logits = self.gate_head(hidden)
        probabilities = F.softmax(logits, dim=1)

        raw_quantiles = self.quantile_head(hidden)
        quantile_median = raw_quantiles[:, 0]
        quantile_lower = (
            quantile_median
            - F.softplus(raw_quantiles[:, 1])
        )
        quantile_upper = (
            quantile_median
            + F.softplus(raw_quantiles[:, 2])
        )

        raw_parameters = torch.stack(
            [head(hidden) for head in self.expert_heads],
            dim=1,
        )

        locations = raw_parameters[:, :, 0]
        scales = (
            self.cfg.minimum_scale
            + (
                self.cfg.maximum_scale
                - self.cfg.minimum_scale
            )
            * torch.sigmoid(raw_parameters[:, :, 1])
        )
        skews = (
            self.cfg.maximum_skew
            * torch.tanh(raw_parameters[:, :, 2])
        )
        tails = (
            self.cfg.minimum_tail
            + (
                self.cfg.maximum_tail
                - self.cfg.minimum_tail
            )
            * torch.sigmoid(raw_parameters[:, :, 3])
        )

        expert_medians = (
            locations
            + scales * torch.sinh(skews / tails)
        )
        mixture_median_approximation = torch.sum(
            probabilities * expert_medians,
            dim=1,
        )

        component_log_probabilities = None
        mixture_log_probability = None

        if target is not None:
            target_values = target.squeeze(-1).unsqueeze(1)
            standardised = (
                target_values - locations
            ) / scales

            inverse_argument = (
                tails * torch.asinh(standardised)
                - skews
            )
            inverse_argument = torch.clamp(
                inverse_argument,
                -12.0,
                12.0,
            )
            base = torch.sinh(inverse_argument)

            base_log_density = (
                -0.5 * base**2
                - 0.5 * math.log(2.0 * math.pi)
            )
            log_jacobian = (
                torch.log(tails)
                - torch.log(scales)
                + stable_log_cosh(inverse_argument)
                - 0.5 * torch.log1p(standardised**2)
            )

            component_log_probabilities = (
                base_log_density + log_jacobian
            )
            mixture_log_probability = torch.logsumexp(
                F.log_softmax(logits, dim=1)
                + component_log_probabilities,
                dim=1,
            )

        return ForwardBundle(
            logits=logits,
            probabilities=probabilities,
            locations=locations,
            scales=scales,
            skews=skews,
            tails=tails,
            expert_medians=expert_medians,
            mixture_median_approximation=(
                mixture_median_approximation
            ),
            quantile_lower=quantile_lower,
            quantile_median=quantile_median,
            quantile_upper=quantile_upper,
            component_log_probabilities=(
                component_log_probabilities
            ),
            mixture_log_probability=mixture_log_probability,
        )

    def sample_from_bundle(
        self,
        bundle: ForwardBundle,
        number_of_samples: int,
    ) -> torch.Tensor:
        batch_size = bundle.locations.shape[0]

        noise = torch.randn(
            batch_size,
            self.cfg.regimes,
            number_of_samples,
            device=bundle.locations.device,
            dtype=bundle.locations.dtype,
        )

        transformed = torch.sinh(
            (
                torch.asinh(noise)
                + bundle.skews.unsqueeze(-1)
            )
            / bundle.tails.unsqueeze(-1)
        )

        component_samples = (
            bundle.locations.unsqueeze(-1)
            + bundle.scales.unsqueeze(-1)
            * transformed
        )

        selected_components = torch.distributions.Categorical(
            probs=bundle.probabilities
        ).sample((number_of_samples,)).transpose(0, 1)

        selected = torch.gather(
            component_samples.permute(0, 2, 1),
            dim=2,
            index=selected_components.unsqueeze(-1),
        ).squeeze(-1)

        return selected

## Consistent training objective, guarded optimisation, and early stopping

The conditional-flow objective is unchanged. Configuration now controls a minimum training phase, the learning-rate schedule, and early stopping independently. Every seed restores its best validation checkpoint; learning rate, pre-clipping gradient norm, epoch time, and total runtime are reported and saved.


In [ ]:
def quantile_loss(
    prediction: torch.Tensor,
    target: torch.Tensor,
    quantile: float,
) -> torch.Tensor:
    error = target - prediction
    return torch.maximum(
        quantile * error,
        (quantile - 1.0) * error,
    ).mean()


def objective(
    model: RegimeFlowMixture,
    context: torch.Tensor,
    target: torch.Tensor,
    labels: torch.Tensor,
    class_weights: torch.Tensor,
    target_gate_usage: torch.Tensor,
    cfg: Config,
) -> tuple[torch.Tensor, dict[str, float]]:
    bundle = model.forward_bundle(context, target)

    if (
        bundle.mixture_log_probability is None
        or bundle.component_log_probabilities is None
    ):
        raise RuntimeError("Training bundle lacks log probabilities.")

    mixture_nll = -bundle.mixture_log_probability.mean()

    selected_expert_nll = (
        -bundle.component_log_probabilities.gather(
            1,
            labels.unsqueeze(1),
        ).mean()
    )

    classification = F.cross_entropy(
        bundle.logits,
        labels,
        weight=class_weights,
        label_smoothing=cfg.label_smoothing,
    )

    average_gate_usage = bundle.probabilities.mean(dim=0)
    gate_balance = torch.mean(
        (
            average_gate_usage
            - target_gate_usage
        ) ** 2
    )

    actual_log_volatility = target.squeeze(1)
    predicted_variance = torch.exp(
        2.0 * bundle.mixture_median_approximation
    ).clamp_min(EPS)
    actual_variance = torch.exp(
        2.0 * actual_log_volatility
    ).clamp_min(EPS)

    variance_ratio = (
        actual_variance / predicted_variance
    )
    qlike = torch.mean(
        variance_ratio
        - torch.log(variance_ratio)
        - 1.0
    )

    quantile_objective = (
        quantile_loss(
            bundle.quantile_lower,
            actual_log_volatility,
            0.05,
        )
        + quantile_loss(
            bundle.quantile_median,
            actual_log_volatility,
            0.50,
        )
        + quantile_loss(
            bundle.quantile_upper,
            actual_log_volatility,
            0.95,
        )
    )

    total = (
        mixture_nll
        + cfg.expert_alignment_weight
        * selected_expert_nll
        + cfg.regime_classification_weight
        * classification
        + cfg.gate_balance_weight
        * gate_balance
        + cfg.qlike_weight * qlike
        + cfg.quantile_weight
        * quantile_objective
    )

    metrics = {
        "total": float(total.detach().cpu()),
        "nll": float(mixture_nll.detach().cpu()),
        "classification": float(
            classification.detach().cpu()
        ),
        "qlike": float(qlike.detach().cpu()),
        "accuracy": float(
            (
                bundle.logits.argmax(dim=1)
                == labels
            )
            .float()
            .mean()
            .detach()
            .cpu()
        ),
        "medium_probability": float(
            bundle.probabilities[:, 1]
            .mean()
            .detach()
            .cpu()
        ),
    }

    return total, metrics


def make_loader(
    split: PreparedSplit,
    dataset: MarketDataset,
    indices: np.ndarray,
    cfg: Config,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    data = TensorDataset(
        torch.from_numpy(
            split.context[indices]
        ).float(),
        torch.from_numpy(
            dataset.targets_log_vol[indices]
        ).float().unsqueeze(1),
        torch.from_numpy(
            split.regime_labels[indices]
        ).long(),
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        data,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        drop_last=False,
        generator=generator,
        num_workers=0,
        pin_memory=(cfg.pin_memory and DEVICE.type == "cuda"),
    )


def run_epoch(
    model: RegimeFlowMixture,
    loader: DataLoader,
    cfg: Config,
    class_weights: torch.Tensor,
    target_gate_usage: torch.Tensor,
    optimiser: optim.Optimizer | None,
) -> dict[str, float]:
    training = optimiser is not None
    model.train(training)

    totals = {}
    observation_count = 0

    for context, target, labels in loader:
        non_blocking = cfg.pin_memory and DEVICE.type == "cuda"
        context = context.to(DEVICE, non_blocking=non_blocking)
        target = target.to(DEVICE, non_blocking=non_blocking)
        labels = labels.to(DEVICE, non_blocking=non_blocking)
        batch_size = len(context)

        if training:
            optimiser.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            total, metrics = objective(
                model,
                context,
                target,
                labels,
                class_weights,
                target_gate_usage,
                cfg,
            )

            if not torch.isfinite(total):
                raise FloatingPointError(
                    "Non-finite objective encountered; training stopped before "
                    "the optimiser could corrupt the checkpoint."
                )

            if training:
                total.backward()
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    cfg.gradient_clip,
                    error_if_nonfinite=True,
                )
                metrics["gradient_norm"] = float(
                    gradient_norm.detach().cpu()
                )
                optimiser.step()

        for key, value in metrics.items():
            totals[key] = (
                totals.get(key, 0.0)
                + value * batch_size
            )
        observation_count += batch_size

    return {
        key: value / observation_count
        for key, value in totals.items()
    }


def fit_single_model(
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
    seed: int,
) -> tuple[
    RegimeFlowMixture,
    dict[str, list[float]],
    int,
]:
    set_seed(seed)

    train_loader = make_loader(
        split,
        dataset,
        split.train_indices,
        cfg,
        shuffle=True,
        seed=seed,
    )
    validation_loader = make_loader(
        split,
        dataset,
        split.validation_indices,
        cfg,
        shuffle=False,
        seed=seed,
    )

    model = RegimeFlowMixture(
        split.context.shape[1],
        cfg,
    ).to(DEVICE)

    optimiser = optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimiser,
        mode="min",
        factor=cfg.scheduler_factor,
        patience=cfg.scheduler_patience,
        threshold=cfg.early_stopping_min_delta,
        threshold_mode="abs",
        cooldown=cfg.scheduler_cooldown,
        min_lr=cfg.minimum_learning_rate,
    )

    class_weights = torch.from_numpy(
        split.class_weights
    ).float().to(DEVICE)
    target_gate_usage = torch.from_numpy(
        split.train_regime_proportions
    ).float().to(DEVICE)

    history = {
        "train_loss": [],
        "validation_loss": [],
        "validation_nll": [],
        "validation_qlike": [],
        "checkpoint_value": [],
        "validation_accuracy": [],
        "validation_medium_probability": [],
        "learning_rate": [],
        "gradient_norm": [],
        "epoch_seconds": [],
    }

    best_checkpoint_value = float("inf")
    best_state = None
    best_epoch = 0
    stale_epochs = 0

    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    print(
        f"\nTraining seed {seed} | parameters={parameter_count:,} | "
        f"minimum/cap epochs={cfg.minimum_epochs}/{cfg.epochs}..."
    )

    for epoch in range(1, cfg.epochs + 1):
        epoch_started = time.perf_counter()
        train_metrics = run_epoch(
            model,
            train_loader,
            cfg,
            class_weights,
            target_gate_usage,
            optimiser,
        )

        with torch.no_grad():
            validation_metrics = run_epoch(
                model,
                validation_loader,
                cfg,
                class_weights,
                target_gate_usage,
                optimiser=None,
            )

        checkpoint_value = (
            validation_metrics["qlike"]
            if cfg.checkpoint_metric == "validation_qlike"
            else validation_metrics["total"]
        )
        scheduler.step(checkpoint_value)
        current_learning_rate = float(optimiser.param_groups[0]["lr"])
        epoch_seconds = time.perf_counter() - epoch_started

        history["train_loss"].append(
            train_metrics["total"]
        )
        history["validation_loss"].append(
            validation_metrics["total"]
        )
        history["validation_nll"].append(
            validation_metrics["nll"]
        )
        history["validation_qlike"].append(
            validation_metrics["qlike"]
        )
        history["checkpoint_value"].append(checkpoint_value)
        history["validation_accuracy"].append(
            validation_metrics["accuracy"]
        )
        history[
            "validation_medium_probability"
        ].append(
            validation_metrics["medium_probability"]
        )
        history["learning_rate"].append(current_learning_rate)
        history["gradient_norm"].append(train_metrics["gradient_norm"])
        history["epoch_seconds"].append(epoch_seconds)

        if (
            checkpoint_value
            < best_checkpoint_value - cfg.early_stopping_min_delta
        ):
            best_checkpoint_value = checkpoint_value
            best_state = copy.deepcopy(
                model.state_dict()
            )
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % cfg.print_every == 0:
            print(
                f"  Epoch {epoch:03d}/{cfg.epochs} | "
                f"train={train_metrics['total']:.4f} | "
                f"validation={validation_metrics['total']:.4f} | "
                f"qlike={validation_metrics['qlike']:.4f} | "
                f"checkpoint={checkpoint_value:.4f} | "
                f"accuracy={validation_metrics['accuracy']:.3f} | "
                f"P(medium)={validation_metrics['medium_probability']:.3f} | "
                f"lr={current_learning_rate:.2e} | "
                f"epoch={epoch_seconds:.1f}s | "
                f"max ETA={np.mean(history['epoch_seconds'][-5:]) * (cfg.epochs - epoch) / 60.0:.1f}m"
            )

        if epoch >= cfg.minimum_epochs and stale_epochs >= cfg.patience:
            print(
                f"  Early stopping; best epoch={best_epoch}"
            )
            break

    if best_state is None:
        raise RuntimeError(
            "Training failed to create a checkpoint."
        )

    model.load_state_dict(best_state)
    model.eval()

    return model, history, best_epoch


def fit_ensemble(
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
) -> tuple[
    list[RegimeFlowMixture],
    list[dict[str, list[float]]],
    list[int],
]:
    models = []
    histories = []
    best_epochs = []

    for seed in cfg.ensemble_seeds:
        model, history, best_epoch = fit_single_model(
            dataset,
            split,
            cfg,
            seed,
        )
        models.append(model)
        histories.append(history)
        best_epochs.append(best_epoch)

    return models, histories, best_epochs


def build_training_summary(
    histories: list[dict[str, list[float]]],
    best_epochs: list[int],
    cfg: Config,
) -> pd.DataFrame:
    rows = []
    for seed, history, best_epoch in zip(
        cfg.ensemble_seeds, histories, best_epochs
    ):
        epochs_run = len(history["validation_loss"])
        best_index = best_epoch - 1
        rows.append({
            "seed": seed,
            "configured_minimum_epochs": cfg.minimum_epochs,
            "configured_epoch_cap": cfg.epochs,
            "epochs_run": epochs_run,
            "best_epoch": best_epoch,
            "checkpoint_metric": cfg.checkpoint_metric,
            "best_checkpoint_value": history["checkpoint_value"][best_index],
            "best_validation_objective": history["validation_loss"][best_index],
            "best_validation_qlike": history["validation_qlike"][best_index],
            "final_validation_objective": history["validation_loss"][-1],
            "final_validation_qlike": history["validation_qlike"][-1],
            "final_learning_rate": history["learning_rate"][-1],
            "final_gradient_norm": history["gradient_norm"][-1],
            "total_seconds": float(np.sum(history["epoch_seconds"])),
            "total_minutes": float(np.sum(history["epoch_seconds"]) / 60.0),
            "stopped_early": epochs_run < cfg.epochs,
        })
    return pd.DataFrame(rows)


## Training-only HAR baselines and ensemble outputs


In [ ]:
def _fit_ridge_har_coefficients(
    dataset: MarketDataset,
    indices: np.ndarray,
    feature_columns: list[int],
) -> np.ndarray:
    design = dataset.features[indices][:, feature_columns].astype(float)
    design = np.column_stack([np.ones(len(design)), design])
    target = dataset.targets_log_vol[indices]
    penalty = 1e-4 * np.eye(design.shape[1])
    penalty[0, 0] = 0.0
    return np.linalg.solve(
        design.T @ design + penalty,
        design.T @ target,
    )


def fit_ticker_specific_har(
    dataset: MarketDataset,
    split: PreparedSplit,
    cfg: Config,
) -> np.ndarray:
    """Use ticker-specific training fits for seen assets and a pooled fit for held-outs."""
    feature_columns = [
        dataset.feature_names.index("log_realised_volatility_5"),
        dataset.feature_names.index("log_realised_volatility_20"),
        dataset.feature_names.index("log_realised_volatility_60"),
    ]
    pooled_coefficients = _fit_ridge_har_coefficients(
        dataset,
        split.train_indices,
        feature_columns,
    )
    predictions = np.full(len(dataset.metadata), np.nan, dtype=float)

    for ticker_id, ticker in enumerate(cfg.target_tickers):
        ticker_all = np.flatnonzero(
            dataset.metadata["ticker_id"].to_numpy(dtype=int) == ticker_id
        )
        if ticker in cfg.training_tickers:
            ticker_train = split.train_indices[
                dataset.metadata.iloc[split.train_indices][
                    "ticker_id"
                ].to_numpy(dtype=int) == ticker_id
            ]
            coefficients = _fit_ridge_har_coefficients(
                dataset,
                ticker_train,
                feature_columns,
            )
        else:
            # No held-out outcome is used to fit this transferable baseline.
            coefficients = pooled_coefficients

        all_design = dataset.features[ticker_all][
            :, feature_columns
        ].astype(float)
        all_design = np.column_stack([
            np.ones(len(all_design)), all_design,
        ])
        predictions[ticker_all] = np.exp(all_design @ coefficients)

    if not np.all(np.isfinite(predictions)):
        raise ValueError("HAR predictions are incomplete.")
    return predictions


def softmax_numpy(
    logits: np.ndarray,
    temperature: float,
) -> np.ndarray:
    scaled = logits / temperature
    scaled = scaled - np.max(scaled, axis=-1, keepdims=True)
    exponentials = np.exp(scaled)
    return exponentials / exponentials.sum(axis=-1, keepdims=True)


@torch.no_grad()
def collect_ensemble_outputs(
    models: list[RegimeFlowMixture],
    dataset: MarketDataset,
    split: PreparedSplit,
    indices: np.ndarray,
    cfg: Config,
) -> dict[str, np.ndarray]:
    # Reset sampling independently of training duration so ablations use the
    # same Monte-Carlo stream and do not gain or lose accuracy by chance.
    set_seed(cfg.prediction_seed)
    samples_by_model = []
    logits_by_model = []
    log_probabilities_by_model = []
    samples_per_model = max(
        32,
        math.ceil(cfg.prediction_samples / len(models)),
    )

    for model in models:
        model_samples = []
        model_logits = []
        model_log_probabilities = []

        for start in range(0, len(indices), cfg.prediction_batch_size):
            batch_indices = indices[start:start + cfg.prediction_batch_size]
            context = torch.from_numpy(
                split.context[batch_indices]
            ).float().to(DEVICE)
            target = torch.from_numpy(
                dataset.targets_log_vol[batch_indices]
            ).float().unsqueeze(1).to(DEVICE)

            bundle = model.forward_bundle(context, target)
            log_samples = model.sample_from_bundle(bundle, samples_per_model)
            if bundle.mixture_log_probability is None:
                raise RuntimeError("Evaluation bundle lacks log probability.")

            model_samples.append(log_samples.cpu().numpy())
            model_logits.append(bundle.logits.cpu().numpy())
            model_log_probabilities.append(
                bundle.mixture_log_probability.cpu().numpy()
            )

        samples_by_model.append(np.concatenate(model_samples, axis=0))
        logits_by_model.append(np.concatenate(model_logits, axis=0))
        log_probabilities_by_model.append(
            np.concatenate(model_log_probabilities, axis=0)
        )

    log_samples = np.concatenate(samples_by_model, axis=1)
    logits = np.stack(logits_by_model, axis=0)
    stacked_log_probabilities = np.stack(
        log_probabilities_by_model,
        axis=1,
    )
    maximum = np.max(stacked_log_probabilities, axis=1, keepdims=True)
    ensemble_log_probability = maximum[:, 0] + np.log(
        np.mean(
            np.exp(stacked_log_probabilities - maximum),
            axis=1,
        ) + EPS
    )

    return {
        "log_samples": log_samples,
        "model_log_samples": np.stack(samples_by_model, axis=0),
        "model_logits": logits,
        "uncalibrated_log_probability": ensemble_log_probability,
    }


## Calibration-only interval scale and regime temperature

In [ ]:
def interval_score(
    lower: np.ndarray,
    upper: np.ndarray,
    actual: np.ndarray,
    alpha: float,
) -> np.ndarray:
    score = upper - lower
    score += (
        2.0 / alpha
        * (lower - actual)
        * (actual < lower)
    )
    score += (
        2.0 / alpha
        * (actual - upper)
        * (actual > upper)
    )
    return score


def calibrate_interval_scale(
    log_samples: np.ndarray,
    actual_log_volatility: np.ndarray,
    cfg: Config,
) -> float:
    median = np.median(
        log_samples,
        axis=1,
        keepdims=True,
    )
    actual = np.exp(actual_log_volatility)

    candidates = np.linspace(
        cfg.interval_scale_min,
        cfg.interval_scale_max,
        cfg.interval_scale_candidates,
    )

    best_scale = 1.0
    best_score = float("inf")

    for scale in candidates:
        adjusted = (
            median
            + scale * (log_samples - median)
        )
        adjusted = np.clip(
            adjusted,
            math.log(cfg.minimum_reported_volatility),
            math.log(cfg.maximum_reported_volatility),
        )
        samples = np.exp(adjusted)

        q05, q25, q75, q95 = np.quantile(
            samples,
            [0.05, 0.25, 0.75, 0.95],
            axis=1,
        )

        score = np.mean(
            interval_score(
                q05,
                q95,
                actual,
                alpha=0.10,
            )
            + interval_score(
                q25,
                q75,
                actual,
                alpha=0.50,
            )
        )

        if score < best_score:
            best_score = score
            best_scale = float(scale)

    print(
        f"Calibration interval scale: {best_scale:.3f}"
    )
    return best_scale


def multiclass_log_loss(
    probabilities: np.ndarray,
    labels: np.ndarray,
) -> float:
    selected = probabilities[
        np.arange(len(labels)),
        labels,
    ]
    return float(
        -np.mean(np.log(selected + EPS))
    )


def calibrate_regime_temperature(
    model_logits: np.ndarray,
    labels: np.ndarray,
    cfg: Config,
) -> float:
    candidates = np.linspace(
        cfg.temperature_min,
        cfg.temperature_max,
        cfg.temperature_candidates,
    )

    best_temperature = 1.0
    best_loss = float("inf")

    for temperature in candidates:
        probabilities = np.mean(
            softmax_numpy(
                model_logits,
                temperature,
            ),
            axis=0,
        )
        loss = multiclass_log_loss(
            probabilities,
            labels,
        )

        if loss < best_loss:
            best_loss = loss
            best_temperature = float(temperature)

    print(
        f"Calibration regime temperature: "
        f"{best_temperature:.3f}"
    )
    return best_temperature


def calibrated_probabilities(
    model_logits: np.ndarray,
    temperature: float,
) -> np.ndarray:
    return np.mean(
        softmax_numpy(
            model_logits,
            temperature,
        ),
        axis=0,
    )

## Predictions and past-only regime smoothing

In [ ]:
def empirical_crps_numpy(
    samples: np.ndarray,
    actual: np.ndarray,
) -> np.ndarray:
    first_term = np.mean(
        np.abs(samples - actual[:, None]),
        axis=1,
    )
    sorted_samples = np.sort(samples, axis=1)
    sample_count = samples.shape[1]
    ranks = np.arange(1, sample_count + 1, dtype=float)
    coefficients = 2.0 * ranks - sample_count - 1.0
    second_term = (
        sorted_samples * coefficients[None, :]
    ).sum(axis=1) / (sample_count**2)
    return first_term - second_term


def regime_probability_columns(prefix: str) -> list[str]:
    return [
        f"{prefix}_{name.lower()}"
        for name in REGIME_NAMES
    ]


def apply_regime_smoothing(
    predictions: pd.DataFrame,
    split: PreparedSplit,
) -> pd.DataFrame:
    predictions = predictions.copy()
    raw_columns = regime_probability_columns("probability")
    smoothed_columns = regime_probability_columns("smoothed_probability")
    for column in smoothed_columns:
        predictions[column] = 0.0

    for ticker_id, ticker_rows in predictions.groupby("ticker_id", sort=False):
        ordered = ticker_rows.sort_values("origin_date")
        posterior = split.initial_regime_probabilities[
            int(ticker_id)
        ].astype(float).copy()
        transition = split.transition_matrices[int(ticker_id)]

        for row_index in ordered.index:
            gate_probability = predictions.loc[
                row_index,
                raw_columns,
            ].to_numpy(dtype=float)
            # Only the training-derived transition, previous filtered state,
            # and current model probability are used.
            prior = posterior @ transition
            posterior = prior * gate_probability
            posterior /= posterior.sum() + EPS
            predictions.loc[row_index, smoothed_columns] = posterior

    predictions["smoothed_regime"] = predictions[
        smoothed_columns
    ].to_numpy().argmax(axis=1)
    predictions["smoothed_regime_name"] = REGIME_NAMES[
        predictions["smoothed_regime"].to_numpy(dtype=int)
    ]
    return predictions


def tolerance_percent_label(tolerance: float) -> int:
    return int(round(100.0 * tolerance))


def add_detailed_result_columns(
    frame: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    frame = frame.copy()
    actual = frame["actual_vol"].to_numpy(dtype=float)
    denominator = np.maximum(np.abs(actual), EPS)

    for model_name, prediction_column in MODEL_COLUMNS.items():
        prefix = MODEL_RESULT_PREFIXES[model_name]
        predicted = frame[prediction_column].to_numpy(dtype=float)
        signed_error = predicted - actual
        absolute_error = np.abs(signed_error)
        absolute_relative_error = absolute_error / denominator
        frame[f"{prefix}_signed_error"] = signed_error
        frame[f"{prefix}_absolute_error"] = absolute_error
        frame[f"{prefix}_absolute_percentage_error"] = (
            100.0 * absolute_relative_error
        )
        frame[f"{prefix}_squared_error"] = signed_error**2
        qlike_prediction = frame[
            QLIKE_MODEL_COLUMNS[model_name]
        ].to_numpy(dtype=float)
        frame[f"{prefix}_qlike_loss"] = qlike_loss_values(
            actual, qlike_prediction
        )
        for tolerance in cfg.success_relative_tolerances:
            label = tolerance_percent_label(tolerance)
            frame[f"{prefix}_within_{label}pct"] = (
                absolute_relative_error <= tolerance
            )

    primary_label = tolerance_percent_label(cfg.primary_success_tolerance)
    frame["success_within_20pct"] = frame["sigflow_within_20pct"]
    frame["success_within_configured_tolerance"] = frame[
        f"sigflow_within_{primary_label}pct"
    ]
    frame["inside_50pct_interval"] = (
        (actual >= frame["predicted_q25_vol"].to_numpy(dtype=float))
        & (actual <= frame["predicted_q75_vol"].to_numpy(dtype=float))
    )
    frame["inside_90pct_interval"] = (
        (actual >= frame["predicted_q05_vol"].to_numpy(dtype=float))
        & (actual <= frame["predicted_q95_vol"].to_numpy(dtype=float))
    )
    frame["interval_width_50"] = (
        frame["predicted_q75_vol"] - frame["predicted_q25_vol"]
    )
    frame["interval_width_90"] = (
        frame["predicted_q95_vol"] - frame["predicted_q05_vol"]
    )
    frame["raw_regime_correct"] = (
        frame["predicted_regime"].to_numpy(dtype=int)
        == frame["regime"].to_numpy(dtype=int)
    )
    frame["smoothed_regime_correct"] = (
        frame["smoothed_regime"].to_numpy(dtype=int)
        == frame["regime"].to_numpy(dtype=int)
    )

    recent = frame["rolling_vol_baseline"].to_numpy(dtype=float)
    frame["actual_direction_vs_recent"] = np.sign(actual - recent).astype(int)
    frame["predicted_direction_vs_recent"] = np.sign(
        frame["predicted_median_vol"].to_numpy(dtype=float) - recent
    ).astype(int)
    frame["direction_correct_vs_recent"] = (
        frame["actual_direction_vs_recent"]
        == frame["predicted_direction_vs_recent"]
    )

    for baseline_prefix in ("rolling", "ewma", "har"):
        frame[f"sigflow_beats_{baseline_prefix}_absolute_error"] = (
            frame["sigflow_absolute_error"]
            < frame[f"{baseline_prefix}_absolute_error"]
        )

    return frame


def build_prediction_frame(
    ensemble_outputs: dict[str, np.ndarray],
    dataset: MarketDataset,
    split: PreparedSplit,
    indices: np.ndarray,
    interval_scale: float,
    regime_temperature: float,
    har_predictions: np.ndarray,
    cfg: Config,
    cohort_override: str | None = None,
) -> pd.DataFrame:
    raw_log_samples = ensemble_outputs["log_samples"]
    median_log = np.median(raw_log_samples, axis=1, keepdims=True)
    calibrated_log_samples = median_log + interval_scale * (
        raw_log_samples - median_log
    )
    calibrated_log_samples = np.clip(
        calibrated_log_samples,
        math.log(cfg.minimum_reported_volatility),
        math.log(cfg.maximum_reported_volatility),
    )
    samples = np.exp(calibrated_log_samples)
    actual = np.exp(dataset.targets_log_vol[indices])
    quantiles = np.quantile(
        samples,
        [0.05, 0.25, 0.50, 0.75, 0.95],
        axis=1,
    )
    probabilities = calibrated_probabilities(
        ensemble_outputs["model_logits"],
        regime_temperature,
    )

    frame = dataset.metadata.iloc[indices].copy().reset_index(drop=True)
    frame["dataset_row_index"] = np.asarray(indices, dtype=int)
    if cohort_override is not None:
        frame["evaluation_cohort"] = cohort_override
    frame["predicted_mean_vol"] = samples.mean(axis=1)
    # The median is the Bayes action for absolute error. QLIKE instead
    # requires expected conditional variance, reported back in vol units.
    frame["predicted_qlike_vol"] = np.sqrt(
        np.mean(samples**2, axis=1)
    )
    frame["predicted_q05_vol"] = quantiles[0]
    frame["predicted_q25_vol"] = quantiles[1]
    frame["predicted_median_vol"] = quantiles[2]
    frame["predicted_q75_vol"] = quantiles[3]
    frame["predicted_q95_vol"] = quantiles[4]
    frame["uncalibrated_negative_log_likelihood"] = -ensemble_outputs[
        "uncalibrated_log_probability"
    ]
    frame["crps"] = empirical_crps_numpy(samples, actual)
    frame["pit"] = np.mean(samples <= actual[:, None], axis=1)
    frame["har_style_baseline"] = har_predictions[indices]
    frame["har_fit_scope"] = np.where(
        frame["ticker"].isin(cfg.training_tickers),
        "ticker_training_only",
        "pooled_training_tickers_only",
    )

    model_log_samples = ensemble_outputs["model_log_samples"]
    for model_index, seed in enumerate(cfg.ensemble_seeds):
        seed_logs = model_log_samples[model_index]
        seed_median = np.median(seed_logs, axis=1, keepdims=True)
        seed_logs = seed_median + interval_scale * (
            seed_logs - seed_median
        )
        seed_samples = np.exp(np.clip(
            seed_logs,
            math.log(cfg.minimum_reported_volatility),
            math.log(cfg.maximum_reported_volatility),
        ))
        frame[f"seed_{seed}_median_vol"] = np.median(
            seed_samples, axis=1
        )
        frame[f"seed_{seed}_qlike_vol"] = np.sqrt(
            np.mean(seed_samples**2, axis=1)
        )

    for regime_id, regime_name in enumerate(REGIME_NAMES):
        frame[f"probability_{regime_name.lower()}"] = probabilities[:, regime_id]
    frame["predicted_regime"] = probabilities.argmax(axis=1)
    frame["predicted_regime_name"] = REGIME_NAMES[
        frame["predicted_regime"].to_numpy(dtype=int)
    ]

    frame = apply_regime_smoothing(frame, split)
    return add_detailed_result_columns(frame, cfg)


## Metric-correct accuracy and dependence-aware significance

Median volatility is used for MAE and tolerance accuracy. QLIKE uses `sqrt(E[volatility²])`, the correct action for conditional variance loss. Inference includes paired moving-date blocks at 10/20/40/60 sessions, a prespecified 20-session primary block, a two-way ticker/date bootstrap, paired within-20% differences, Holm-corrected secondary tests, and per-seed stability. More bootstrap repetitions stabilise estimates; they do not manufacture power.


In [ ]:
MODEL_COLUMNS = {
    "SigFlow v4": "predicted_median_vol",
    "Rolling volatility": "rolling_vol_baseline",
    "EWMA volatility": "ewma_vol_baseline",
    "HAR-style transferable": "har_style_baseline",
}
MODEL_RESULT_PREFIXES = {
    "SigFlow v4": "sigflow",
    "Rolling volatility": "rolling",
    "EWMA volatility": "ewma",
    "HAR-style transferable": "har",
}
QLIKE_MODEL_COLUMNS = {
    "SigFlow v4": "predicted_qlike_vol",
    "Rolling volatility": "rolling_vol_baseline",
    "EWMA volatility": "ewma_vol_baseline",
    "HAR-style transferable": "har_style_baseline",
}


def qlike_loss_values(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> np.ndarray:
    actual_variance = np.maximum(actual**2, EPS)
    predicted_variance = np.maximum(predicted**2, EPS)
    ratio = actual_variance / predicted_variance
    return ratio - np.log(ratio) - 1.0


def qlike(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> float:
    return float(np.mean(qlike_loss_values(actual, predicted)))


def point_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
    cfg: Config,
    qlike_predicted: np.ndarray | None = None,
) -> dict[str, float]:
    if qlike_predicted is None:
        qlike_predicted = predicted
    error = predicted - actual
    absolute_error = np.abs(error)
    absolute_relative_error = absolute_error / np.maximum(np.abs(actual), EPS)
    correlation = (
        float(np.corrcoef(actual, predicted)[0, 1])
        if np.std(actual) > EPS and np.std(predicted) > EPS
        else float("nan")
    )
    total_square = float(np.sum((actual - np.mean(actual)) ** 2))
    metrics = {
        "mae": float(np.mean(absolute_error)),
        "rmse": float(np.sqrt(np.mean(error**2))),
        "bias": float(np.mean(error)),
        "mape_percent": float(100.0 * np.mean(absolute_relative_error)),
        "median_ape_percent": float(100.0 * np.median(absolute_relative_error)),
        "smape_percent": float(100.0 * np.mean(
            2.0 * absolute_error
            / np.maximum(np.abs(actual) + np.abs(predicted), EPS)
        )),
        "correlation": correlation,
        "r_squared": (
            float(1.0 - np.sum(error**2) / total_square)
            if total_square > EPS else float("nan")
        ),
        "qlike": qlike(actual, qlike_predicted),
    }
    for tolerance in cfg.success_relative_tolerances:
        label = tolerance_percent_label(tolerance)
        metrics[f"within_{label}pct_accuracy"] = float(
            np.mean(absolute_relative_error <= tolerance)
        )
    return metrics


def confusion_matrix_numpy(
    actual: np.ndarray,
    predicted: np.ndarray,
    classes: int,
) -> np.ndarray:
    matrix = np.zeros((classes, classes), dtype=int)
    for actual_value, predicted_value in zip(actual, predicted):
        matrix[int(actual_value), int(predicted_value)] += 1
    return matrix


def regime_metrics(
    labels: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
    classes: int,
) -> tuple[dict[str, float], np.ndarray]:
    matrix = confusion_matrix_numpy(labels, predictions, classes)
    recall = np.diag(matrix) / np.maximum(matrix.sum(axis=1), 1)
    precision = np.diag(matrix) / np.maximum(matrix.sum(axis=0), 1)
    f1 = 2.0 * precision * recall / np.maximum(precision + recall, EPS)
    one_hot = np.eye(classes)[labels]
    brier = float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))
    metrics = {
        "accuracy": float(np.mean(labels == predictions)),
        "balanced_accuracy": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "brier_score": brier,
        "log_loss": multiclass_log_loss(probabilities, labels),
    }
    for regime_id, regime_name in enumerate(REGIME_NAMES):
        metrics[f"recall_{regime_name.lower()}"] = float(recall[regime_id])
    return metrics, matrix


def evaluation_slices(
    predictions: pd.DataFrame,
    include_tickers: bool = True,
):
    yield {
        "group_level": "pooled",
        "group": "Pooled",
        "cohort": "all",
        "ticker": "all",
        "rows": predictions,
    }
    for cohort, cohort_rows in predictions.groupby(
        "evaluation_cohort", sort=True
    ):
        yield {
            "group_level": "cohort",
            "group": cohort,
            "cohort": cohort,
            "ticker": "all",
            "rows": cohort_rows,
        }
        for sector, sector_rows in cohort_rows.groupby("sector", sort=True):
            yield {
                "group_level": "cohort_sector",
                "group": f"{cohort}::{sector}",
                "cohort": cohort,
                "ticker": "all",
                "rows": sector_rows,
            }
        if include_tickers:
            for ticker, ticker_rows in cohort_rows.groupby("ticker", sort=True):
                yield {
                    "group_level": "cohort_ticker",
                    "group": f"{cohort}::{ticker}",
                    "cohort": cohort,
                    "ticker": ticker,
                    "rows": ticker_rows,
                }


def evaluate_point_models(
    predictions: pd.DataFrame,
    group_information: dict[str, object],
    cfg: Config,
) -> pd.DataFrame:
    rows = []
    actual = predictions["actual_vol"].to_numpy(dtype=float)
    for model_name, column in MODEL_COLUMNS.items():
        row = {
            key: group_information[key]
            for key in ("group_level", "group", "cohort", "ticker")
        }
        row.update({
            "model": model_name,
            "observations": len(predictions),
        })
        row.update(point_metrics(
            actual,
            predictions[column].to_numpy(dtype=float),
            cfg,
            qlike_predicted=predictions[
                QLIKE_MODEL_COLUMNS[model_name]
            ].to_numpy(dtype=float),
        ))
        row["point_forecast_column"] = column
        row["qlike_forecast_column"] = QLIKE_MODEL_COLUMNS[model_name]
        rows.append(row)
    return pd.DataFrame(rows)


def evaluate_all_groups(
    predictions: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    tables = []
    for information in evaluation_slices(predictions, include_tickers=True):
        tables.append(evaluate_point_models(
            information["rows"], information, cfg
        ))
    return pd.concat(tables, ignore_index=True)


def build_accuracy_summary(
    predictions: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    rows = []
    for information in evaluation_slices(predictions, include_tickers=True):
        selected = information["rows"]
        row = {
            key: information[key]
            for key in ("group_level", "group", "cohort", "ticker")
        }
        row["observations"] = len(selected)
        row["successful_forecasts_20pct"] = int(
            selected["success_within_20pct"].sum()
        )
        for model_name in MODEL_COLUMNS:
            prefix = MODEL_RESULT_PREFIXES[model_name]
            for tolerance in cfg.success_relative_tolerances:
                label = tolerance_percent_label(tolerance)
                row[f"{prefix}_within_{label}pct_accuracy"] = float(
                    selected[f"{prefix}_within_{label}pct"].mean()
                )
        row["sigflow_direction_accuracy"] = float(
            selected["direction_correct_vs_recent"].mean()
        )
        row["coverage_50"] = float(selected["inside_50pct_interval"].mean())
        row["coverage_90"] = float(selected["inside_90pct_interval"].mean())
        row["raw_regime_accuracy"] = float(selected["raw_regime_correct"].mean())
        row["smoothed_regime_accuracy"] = float(
            selected["smoothed_regime_correct"].mean()
        )
        for baseline_prefix in ("rolling", "ewma", "har"):
            row[f"sigflow_beats_{baseline_prefix}_rate"] = float(
                selected[
                    f"sigflow_beats_{baseline_prefix}_absolute_error"
                ].mean()
            )
        rows.append(row)
    return pd.DataFrame(rows)


def build_daily_success_summary(
    predictions: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    rows = []
    for (cohort, origin_date), selected in predictions.groupby(
        ["evaluation_cohort", "origin_date"], sort=True
    ):
        successful = selected.loc[
            selected["success_within_20pct"], "ticker"
        ].astype(str).tolist()
        failed = selected.loc[
            ~selected["success_within_20pct"], "ticker"
        ].astype(str).tolist()
        row = {
            "cohort": cohort,
            "origin_date": pd.Timestamp(origin_date),
            "forecast_count": len(selected),
            "success_count_20pct": len(successful),
            "success_rate_20pct": float(
                selected["success_within_20pct"].mean()
            ),
            "any_success_20pct": bool(successful),
            "all_success_20pct": not bool(failed),
            "successful_tickers": ", ".join(successful),
            "failed_tickers": ", ".join(failed),
            "sigflow_mae": float(selected["sigflow_absolute_error"].mean()),
            "sigflow_mape_percent": float(
                selected["sigflow_absolute_percentage_error"].mean()
            ),
            "coverage_90": float(selected["inside_90pct_interval"].mean()),
            "raw_regime_accuracy": float(selected["raw_regime_correct"].mean()),
            "smoothed_regime_accuracy": float(
                selected["smoothed_regime_correct"].mean()
            ),
        }
        for tolerance in cfg.success_relative_tolerances:
            label = tolerance_percent_label(tolerance)
            row[f"within_{label}pct_accuracy"] = float(
                selected[f"sigflow_within_{label}pct"].mean()
            )
        rows.append(row)
    return pd.DataFrame(rows).sort_values(
        ["origin_date", "cohort"]
    ).reset_index(drop=True)


def probabilistic_summary_by_group(
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for information in evaluation_slices(predictions, include_tickers=False):
        selected = information["rows"]
        row = {
            key: information[key]
            for key in ("group_level", "group", "cohort", "ticker")
        }
        row.update({
            "observations": len(selected),
            "uncalibrated_nll": float(
                selected["uncalibrated_negative_log_likelihood"].mean()
            ),
            "crps": float(selected["crps"].mean()),
            "coverage_50": float(selected["inside_50pct_interval"].mean()),
            "coverage_90": float(selected["inside_90pct_interval"].mean()),
            "interval_width_50": float(selected["interval_width_50"].mean()),
            "interval_width_90": float(selected["interval_width_90"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def regime_summary_by_group(
    predictions: pd.DataFrame,
    cfg: Config,
    smoothed: bool,
) -> tuple[pd.DataFrame, np.ndarray, dict[str, np.ndarray]]:
    rows = []
    matrices = {}
    probability_prefix = "smoothed_probability" if smoothed else "probability"
    prediction_column = "smoothed_regime" if smoothed else "predicted_regime"
    pooled_matrix = np.zeros((cfg.regimes, cfg.regimes), dtype=int)

    for information in evaluation_slices(predictions, include_tickers=False):
        selected = information["rows"]
        labels = selected["regime"].to_numpy(dtype=int)
        predicted = selected[prediction_column].to_numpy(dtype=int)
        probabilities = selected[
            regime_probability_columns(probability_prefix)
        ].to_numpy(dtype=float)
        metrics, matrix = regime_metrics(
            labels, predicted, probabilities, cfg.regimes
        )
        row = {
            key: information[key]
            for key in ("group_level", "group", "cohort", "ticker")
        }
        row["observations"] = len(selected)
        row.update(metrics)
        rows.append(row)
        matrices[str(information["group"])] = matrix
        if information["group_level"] == "pooled":
            pooled_matrix = matrix

    return pd.DataFrame(rows), pooled_matrix, matrices


def non_overlapping_phase_metrics(
    predictions: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    tables = []
    for information in evaluation_slices(predictions, include_tickers=False):
        cohort_rows = information["rows"]
        for phase in range(cfg.horizon):
            selected_pieces = []
            for _, ticker_rows in cohort_rows.groupby("ticker", sort=False):
                ordered = ticker_rows.sort_values("origin_date")
                selected_pieces.append(ordered.iloc[phase::cfg.horizon])
            selected = pd.concat(selected_pieces, ignore_index=True)
            phase_information = dict(information)
            phase_information["rows"] = selected
            table = evaluate_point_models(selected, phase_information, cfg)
            table["phase"] = phase
            tables.append(table)
    return pd.concat(tables, ignore_index=True)


def moving_block_draw(
    date_count: int,
    requested_block_length: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, int]:
    effective = min(requested_block_length, date_count)
    possible_starts = np.arange(max(1, date_count - effective + 1))
    selected = []
    while len(selected) < date_count:
        start = int(rng.choice(possible_starts))
        selected.extend(range(start, start + effective))
    return np.asarray(selected[:date_count], dtype=int), effective


def paired_difference_arrays(
    predictions: pd.DataFrame,
    baseline_name: str,
    cfg: Config,
) -> dict[str, np.ndarray]:
    actual = predictions["actual_vol"].to_numpy(dtype=float)
    sigflow_point = predictions[MODEL_COLUMNS["SigFlow v4"]].to_numpy(dtype=float)
    sigflow_qlike = predictions[QLIKE_MODEL_COLUMNS["SigFlow v4"]].to_numpy(dtype=float)
    baseline_point = predictions[MODEL_COLUMNS[baseline_name]].to_numpy(dtype=float)
    baseline_qlike = predictions[QLIKE_MODEL_COLUMNS[baseline_name]].to_numpy(dtype=float)
    tolerance = cfg.primary_success_tolerance
    denominator = np.maximum(np.abs(actual), EPS)
    return {
        "mae": np.abs(sigflow_point - actual) - np.abs(baseline_point - actual),
        "qlike": (
            qlike_loss_values(actual, sigflow_qlike)
            - qlike_loss_values(actual, baseline_qlike)
        ),
        "within_20pct_accuracy": (
            (np.abs(sigflow_point - actual) / denominator <= tolerance).astype(float)
            - (np.abs(baseline_point - actual) / denominator <= tolerance).astype(float)
        ),
    }


def bootstrap_result_row(
    values: np.ndarray,
    observed: float,
    cohort: str,
    baseline: str,
    metric: str,
    scheme: str,
    requested_block: int,
    effective_block: int,
    date_count: int,
    ticker_count: int,
    repetitions: int,
    cfg: Config,
) -> dict[str, object]:
    higher_is_better = metric == "within_20pct_accuracy"
    if len(values) < 2 or not np.all(np.isfinite(values)):
        return {
            "cohort": cohort,
            "comparison": f"SigFlow v4 minus {baseline}",
            "metric": metric,
            "resampling_scheme": scheme,
            "requested_block_length": requested_block,
            "effective_block_length": effective_block,
            "unique_dates": date_count,
            "ticker_clusters": ticker_count,
            "approximate_effective_blocks": date_count / max(effective_block, 1),
            "repetitions": repetitions,
            "observed_difference": observed,
            "mean_difference": float("nan"),
            "standard_error": float("nan"),
            "lower_95": float("nan"),
            "upper_95": float("nan"),
            "raw_p_value": float("nan"),
            "sigflow_better_if": "difference > 0" if higher_is_better else "difference < 0",
            "inference_available": False,
            "unavailable_reason": "fewer than two valid bootstrap replicates",
            "is_primary_block_length": requested_block == cfg.bootstrap_block_days,
        }
    centered = values - np.mean(values)
    raw_p = (
        1.0 + np.sum(np.abs(centered) >= abs(observed))
    ) / (len(values) + 1.0)
    return {
        "cohort": cohort,
        "comparison": f"SigFlow v4 minus {baseline}",
        "metric": metric,
        "resampling_scheme": scheme,
        "requested_block_length": requested_block,
        "effective_block_length": effective_block,
        "unique_dates": date_count,
        "ticker_clusters": ticker_count,
        "approximate_effective_blocks": date_count / max(effective_block, 1),
        "repetitions": repetitions,
        "observed_difference": observed,
        "mean_difference": float(np.mean(values)),
        "standard_error": float(np.std(values, ddof=1)),
        "lower_95": float(np.quantile(values, 0.025)),
        "upper_95": float(np.quantile(values, 0.975)),
        "raw_p_value": float(raw_p),
        "sigflow_better_if": "difference > 0" if higher_is_better else "difference < 0",
        "inference_available": True,
        "unavailable_reason": "",
        "is_primary_block_length": requested_block == cfg.bootstrap_block_days,
    }


def paired_block_bootstrap(
    predictions: pd.DataFrame,
    cfg: Config,
    cohort: str,
    requested_block_length: int,
    repetitions: int,
) -> pd.DataFrame:
    ordered_dates = np.array(sorted(pd.to_datetime(
        predictions["origin_date"]
    ).unique()))
    date_codes = pd.Categorical(
        pd.to_datetime(predictions["origin_date"]),
        categories=ordered_dates,
        ordered=True,
    ).codes
    rng = np.random.default_rng(
        cfg.seed + 1009 * requested_block_length + len(predictions)
    )
    draws = []
    effective_block = min(requested_block_length, len(ordered_dates))
    for _ in range(repetitions):
        draw, effective_block = moving_block_draw(
            len(ordered_dates), requested_block_length, rng
        )
        draws.append(draw)
    draws = np.asarray(draws, dtype=int)

    rows = []
    baseline_names = [name for name in MODEL_COLUMNS if name != "SigFlow v4"]
    counts = np.bincount(date_codes, minlength=len(ordered_dates)).astype(float)
    for baseline in baseline_names:
        differences = paired_difference_arrays(predictions, baseline, cfg)
        for metric, row_differences in differences.items():
            sums = np.bincount(
                date_codes,
                weights=row_differences,
                minlength=len(ordered_dates),
            )
            bootstrap_values = np.array([
                sums[draw].sum() / max(counts[draw].sum(), 1.0)
                for draw in draws
            ])
            rows.append(bootstrap_result_row(
                bootstrap_values,
                float(np.mean(row_differences)),
                cohort,
                baseline,
                metric,
                "date_block",
                requested_block_length,
                effective_block,
                len(ordered_dates),
                predictions["ticker"].nunique(),
                repetitions,
                cfg,
            ))
    return pd.DataFrame(rows)


def paired_ticker_date_bootstrap(
    predictions: pd.DataFrame,
    cfg: Config,
    cohort: str,
) -> pd.DataFrame:
    tickers = sorted(predictions["ticker"].astype(str).unique())
    dates = np.array(sorted(pd.to_datetime(
        predictions["origin_date"]
    ).unique()))
    panel_counts = predictions.groupby(["ticker", "origin_date"]).size()
    complete = (
        len(panel_counts) == len(tickers) * len(dates)
        and int(panel_counts.min()) == 1
        and int(panel_counts.max()) == 1
    )
    if not complete or len(tickers) < 2 or len(dates) < 2:
        return pd.DataFrame([{
            "cohort": cohort,
            "comparison": f"SigFlow v4 minus {cfg.primary_baseline}",
            "metric": cfg.primary_metric,
            "resampling_scheme": "ticker_date_two_way",
            "requested_block_length": cfg.bootstrap_block_days,
            "effective_block_length": min(cfg.bootstrap_block_days, len(dates)),
            "unique_dates": len(dates),
            "ticker_clusters": len(tickers),
            "approximate_effective_blocks": len(dates) / max(cfg.bootstrap_block_days, 1),
            "repetitions": 0,
            "observed_difference": float("nan"),
            "mean_difference": float("nan"),
            "standard_error": float("nan"),
            "lower_95": float("nan"),
            "upper_95": float("nan"),
            "raw_p_value": float("nan"),
            "sigflow_better_if": "difference < 0",
            "inference_available": False,
            "unavailable_reason": "ticker/date panel is incomplete or too small",
            "is_primary_block_length": True,
        }])

    ordered = predictions.copy()
    ordered["origin_date"] = pd.to_datetime(ordered["origin_date"])
    ordered = ordered.set_index(["ticker", "origin_date"])
    rng = np.random.default_rng(cfg.seed + 987654)
    ticker_draws = rng.integers(
        0, len(tickers),
        size=(cfg.ticker_date_bootstrap_repetitions, len(tickers)),
    )
    date_draws = []
    effective_block = min(cfg.bootstrap_block_days, len(dates))
    for _ in range(cfg.ticker_date_bootstrap_repetitions):
        draw, effective_block = moving_block_draw(
            len(dates), cfg.bootstrap_block_days, rng
        )
        date_draws.append(draw)

    rows = []
    for baseline in [name for name in MODEL_COLUMNS if name != "SigFlow v4"]:
        flat = predictions.copy()
        differences = paired_difference_arrays(flat, baseline, cfg)
        for metric, values in differences.items():
            value_frame = flat[["ticker", "origin_date"]].copy()
            value_frame["difference"] = values
            matrix = value_frame.pivot(
                index="ticker", columns="origin_date", values="difference"
            ).reindex(index=tickers, columns=dates).to_numpy(dtype=float)
            if not np.all(np.isfinite(matrix)):
                bootstrap_values = np.array([], dtype=float)
            else:
                bootstrap_values = np.array([
                    matrix[np.ix_(ticker_draws[index], date_draws[index])].mean()
                    for index in range(cfg.ticker_date_bootstrap_repetitions)
                ])
            rows.append(bootstrap_result_row(
                bootstrap_values,
                float(np.mean(values)),
                cohort,
                baseline,
                metric,
                "ticker_date_two_way",
                cfg.bootstrap_block_days,
                effective_block,
                len(dates),
                len(tickers),
                cfg.ticker_date_bootstrap_repetitions,
                cfg,
            ))
    return pd.DataFrame(rows)


def apply_holm_correction(
    inference: pd.DataFrame,
    cfg: Config,
) -> pd.DataFrame:
    inference = inference.copy()
    inference["holm_adjusted_p_value"] = np.nan
    inference["raw_significant_0_05"] = (
        inference["raw_p_value"] < cfg.significance_alpha
    )
    inference["holm_significant_0_05"] = False
    inference["holm_family"] = "outside_prespecified_secondary_family"
    family_mask = (
        (inference["resampling_scheme"] == "ticker_date_two_way")
        & inference["cohort"].isin([
            "seen_ticker_unseen_day", "unseen_ticker_unseen_day"
        ])
        & inference["metric"].isin(["mae", "qlike", "within_20pct_accuracy"])
        & np.isfinite(inference["raw_p_value"])
    )
    family_indices = inference.index[family_mask].tolist()
    if family_indices:
        ordered = sorted(
            family_indices,
            key=lambda index: float(inference.loc[index, "raw_p_value"]),
        )
        family_size = len(ordered)
        running = 0.0
        for rank, index in enumerate(ordered):
            adjusted = min(
                1.0,
                (family_size - rank)
                * float(inference.loc[index, "raw_p_value"]),
            )
            running = max(running, adjusted)
            inference.loc[index, "holm_adjusted_p_value"] = running
        inference.loc[family_indices, "holm_family"] = (
            "two_final_cohorts_x_three_baselines_x_three_metrics"
        )
        inference.loc[family_indices, "holm_significant_0_05"] = (
            inference.loc[family_indices, "holm_adjusted_p_value"]
            < cfg.significance_alpha
        )
    return inference


def bootstrap_evidence_label(
    mean_difference: float,
    lower_95: float,
    upper_95: float,
    higher_is_better: bool = False,
) -> str:
    if not np.all(np.isfinite([mean_difference, lower_95, upper_95])):
        return "bootstrap_not_run"
    if higher_is_better:
        if lower_95 > 0.0:
            return "strong_evidence_sigflow_better"
        if upper_95 < 0.0:
            return "strong_evidence_sigflow_worse"
        return (
            "mixed_ci_direction_sigflow"
            if mean_difference > 0.0
            else "mixed_ci_direction_baseline"
        )
    if upper_95 < 0.0:
        return "strong_evidence_sigflow_better"
    if lower_95 > 0.0:
        return "strong_evidence_sigflow_worse"
    return (
        "mixed_ci_direction_sigflow"
        if mean_difference < 0.0
        else "mixed_ci_direction_baseline"
    )


def build_per_seed_metrics(
    predictions: pd.DataFrame,
    cfg: Config,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    group_values = [("all", predictions)] + list(
        predictions.groupby("evaluation_cohort", sort=True)
    )
    primary_label = tolerance_percent_label(cfg.primary_success_tolerance)
    for cohort, selected in group_values:
        actual = selected["actual_vol"].to_numpy(dtype=float)
        har = selected["har_style_baseline"].to_numpy(dtype=float)
        har_metrics = point_metrics(actual, har, cfg, qlike_predicted=har)
        for seed in cfg.ensemble_seeds:
            median = selected[f"seed_{seed}_median_vol"].to_numpy(dtype=float)
            qlike_forecast = selected[f"seed_{seed}_qlike_vol"].to_numpy(dtype=float)
            metrics = point_metrics(
                actual, median, cfg, qlike_predicted=qlike_forecast
            )
            qlike_difference = metrics["qlike"] - har_metrics["qlike"]
            rows.append({
                "cohort": cohort,
                "seed": seed,
                "observations": len(selected),
                "mae": metrics["mae"],
                "qlike": metrics["qlike"],
                f"within_{primary_label}pct_accuracy": metrics[
                    f"within_{primary_label}pct_accuracy"
                ],
                "mae_difference_vs_har": metrics["mae"] - har_metrics["mae"],
                "qlike_difference_vs_har": qlike_difference,
                "qlike_relative_improvement_vs_har": (
                    (har_metrics["qlike"] - metrics["qlike"])
                    / max(abs(har_metrics["qlike"]), EPS)
                ),
                "within_primary_difference_vs_har": (
                    metrics[f"within_{primary_label}pct_accuracy"]
                    - har_metrics[f"within_{primary_label}pct_accuracy"]
                ),
                "qlike_favours_sigflow": qlike_difference < 0.0,
            })
    per_seed = pd.DataFrame(rows)
    stability_rows = []
    for cohort, selected in per_seed.groupby("cohort", sort=True):
        prediction_rows = predictions if cohort == "all" else predictions[
            predictions["evaluation_cohort"] == cohort
        ]
        seed_columns = [
            f"seed_{seed}_median_vol" for seed in cfg.ensemble_seeds
        ]
        row_dispersion = prediction_rows[seed_columns].std(axis=1, ddof=0)
        qlike_differences = selected["qlike_difference_vs_har"].to_numpy(dtype=float)
        stability_rows.append({
            "cohort": cohort,
            "seed_count": len(selected),
            "fraction_seeds_qlike_favours_sigflow": float(
                np.mean(qlike_differences < 0.0)
            ),
            "all_seeds_qlike_favour_sigflow": bool(
                np.all(qlike_differences < 0.0)
            ),
            "qlike_difference_mean": float(np.mean(qlike_differences)),
            "qlike_difference_std": float(np.std(qlike_differences)),
            "qlike_difference_min": float(np.min(qlike_differences)),
            "qlike_difference_max": float(np.max(qlike_differences)),
            "row_forecast_dispersion_mean": float(row_dispersion.mean()),
            "row_forecast_dispersion_p90": float(row_dispersion.quantile(0.90)),
            "stability_sufficient": len(selected) >= 3,
        })
    return per_seed, pd.DataFrame(stability_rows)


def build_primary_claim_summary(
    point_table: pd.DataFrame,
    inference: pd.DataFrame | None,
    seed_stability: pd.DataFrame,
    predictions: pd.DataFrame,
    cfg: Config,
    data_source: str,
    evaluation_split: str,
) -> pd.DataFrame:
    cohort = cfg.primary_evaluation_cohort
    selected_points = point_table[
        (point_table["group_level"] == "cohort")
        & (point_table["group"] == cohort)
    ]
    sigflow = selected_points[selected_points["model"] == "SigFlow v4"]
    baseline = selected_points[selected_points["model"] == cfg.primary_baseline]
    relative_improvement = float("nan")
    if not sigflow.empty and not baseline.empty:
        relative_improvement = float(
            (baseline.iloc[0]["qlike"] - sigflow.iloc[0]["qlike"])
            / max(abs(float(baseline.iloc[0]["qlike"])), EPS)
        )
    primary_row = pd.DataFrame()
    if inference is not None:
        primary_row = inference[
            (inference["cohort"] == cohort)
            & (inference["comparison"] == f"SigFlow v4 minus {cfg.primary_baseline}")
            & (inference["metric"] == cfg.primary_metric)
            & (inference["resampling_scheme"] == "ticker_date_two_way")
        ]
    if primary_row.empty:
        observed = lower = upper = raw_p = holm_p = float("nan")
        inference_available = False
    else:
        selected = primary_row.iloc[0]
        observed = float(selected["observed_difference"])
        lower = float(selected["lower_95"])
        upper = float(selected["upper_95"])
        raw_p = float(selected["raw_p_value"])
        holm_p = float(selected["holm_adjusted_p_value"])
        inference_available = bool(selected["inference_available"])
    sensitivity_rows = pd.DataFrame()
    if inference is not None:
        sensitivity_rows = inference[
            (inference["cohort"] == cohort)
            & (inference["comparison"] == f"SigFlow v4 minus {cfg.primary_baseline}")
            & (inference["metric"] == cfg.primary_metric)
            & (inference["resampling_scheme"] == "date_block")
        ]
    block_sensitivity_robust = (
        not sensitivity_rows.empty
        and bool(np.all(
            sensitivity_rows["observed_difference"].to_numpy(dtype=float) < 0.0
        ))
    )
    seed_row = seed_stability[seed_stability["cohort"] == cohort]
    seed_fraction = (
        float(seed_row.iloc[0]["fraction_seeds_qlike_favours_sigflow"])
        if not seed_row.empty else float("nan")
    )
    cohort_rows = predictions[predictions["evaluation_cohort"] == cohort]
    unique_dates = int(cohort_rows["origin_date"].nunique())
    ticker_count = int(cohort_rows["ticker"].nunique())
    real_final = (
        evaluation_split == "test"
        and not str(data_source).startswith("synthetic")
    )
    adequate_panel = (
        unique_dates >= cfg.minimum_test_origins_per_ticker
        and ticker_count >= 15
    )
    statistical = inference_available and upper < 0.0 and raw_p < cfg.significance_alpha
    practical = relative_improvement >= cfg.primary_min_relative_improvement
    stable = seed_fraction >= cfg.minimum_seed_improvement_fraction
    if not real_final:
        verdict = "pipeline_validation_only"
    elif not adequate_panel:
        verdict = "underpowered_panel"
    elif statistical and practical and stable and block_sensitivity_robust:
        verdict = "supported"
    else:
        verdict = "not_supported"
    return pd.DataFrame([{
        "claim": "SigFlow improves unseen-ticker QLIKE versus HAR",
        "evaluation_cohort": cohort,
        "baseline": cfg.primary_baseline,
        "metric": cfg.primary_metric,
        "data_source": data_source,
        "evaluation_split": evaluation_split,
        "unique_test_origins": unique_dates,
        "unseen_ticker_count": ticker_count,
        "observed_difference": observed,
        "lower_95": lower,
        "upper_95": upper,
        "raw_p_value": raw_p,
        "holm_adjusted_p_value_secondary_family": holm_p,
        "relative_qlike_improvement": relative_improvement,
        "required_relative_improvement": cfg.primary_min_relative_improvement,
        "fraction_seeds_favouring_sigflow": seed_fraction,
        "required_seed_fraction": cfg.minimum_seed_improvement_fraction,
        "real_untouched_final_test": real_final,
        "adequately_powered_panel": adequate_panel,
        "statistical_requirement_met": statistical,
        "practical_requirement_met": practical,
        "seed_stability_requirement_met": stable,
        "all_block_sensitivities_favour_sigflow": block_sensitivity_robust,
        "verdict": verdict,
    }])


def build_baseline_comparison_summary(
    point_table: pd.DataFrame,
    accuracy_summary: pd.DataFrame,
    bootstrap: pd.DataFrame | None,
    cfg: Config,
) -> pd.DataFrame:
    rows = []
    primary_label = tolerance_percent_label(cfg.primary_success_tolerance)
    group_keys = point_table.loc[
        point_table["group_level"].isin(["pooled", "cohort"]),
        ["group_level", "group", "cohort", "ticker"],
    ].drop_duplicates()

    for group in group_keys.to_dict("records"):
        mask = (
            (point_table["group_level"] == group["group_level"])
            & (point_table["group"] == group["group"])
        )
        group_points = point_table.loc[mask]
        sigflow = group_points.loc[
            group_points["model"] == "SigFlow v4"
        ].iloc[0]
        accuracy = accuracy_summary.loc[
            (accuracy_summary["group_level"] == group["group_level"])
            & (accuracy_summary["group"] == group["group"])
        ].iloc[0]
        bootstrap_cohort = "all" if group["group_level"] == "pooled" else group["group"]

        for baseline_name in MODEL_COLUMNS:
            if baseline_name == "SigFlow v4":
                continue
            baseline = group_points.loc[
                group_points["model"] == baseline_name
            ].iloc[0]
            prefix = MODEL_RESULT_PREFIXES[baseline_name]
            row = {
                **group,
                "observations": int(sigflow["observations"]),
                "baseline": baseline_name,
                "sigflow_mae": float(sigflow["mae"]),
                "baseline_mae": float(baseline["mae"]),
                "mae_difference": float(sigflow["mae"] - baseline["mae"]),
                "mae_relative_improvement_percent": float(
                    100.0 * (baseline["mae"] - sigflow["mae"])
                    / max(abs(float(baseline["mae"])), EPS)
                ),
                "sigflow_qlike": float(sigflow["qlike"]),
                "baseline_qlike": float(baseline["qlike"]),
                "qlike_difference": float(sigflow["qlike"] - baseline["qlike"]),
                "qlike_relative_improvement_percent": float(
                    100.0 * (baseline["qlike"] - sigflow["qlike"])
                    / max(abs(float(baseline["qlike"])), EPS)
                ),
            }
            for tolerance in cfg.success_relative_tolerances:
                label = tolerance_percent_label(tolerance)
                sigflow_accuracy = float(accuracy[f"sigflow_within_{label}pct_accuracy"])
                baseline_accuracy = float(accuracy[f"{prefix}_within_{label}pct_accuracy"])
                row[f"sigflow_within_{label}pct_accuracy"] = sigflow_accuracy
                row[f"baseline_within_{label}pct_accuracy"] = baseline_accuracy
                row[f"accuracy_difference_{label}pct_percentage_points"] = (
                    100.0 * (sigflow_accuracy - baseline_accuracy)
                )

            for metric in ("mae", "qlike"):
                bootstrap_row = pd.DataFrame()
                if bootstrap is not None:
                    preferred_scheme = (
                        "date_block"
                        if group["group_level"] == "pooled"
                        else "ticker_date_two_way"
                    )
                    bootstrap_row = bootstrap.loc[
                        (bootstrap["cohort"] == bootstrap_cohort)
                        & (bootstrap["comparison"] == f"SigFlow v4 minus {baseline_name}")
                        & (bootstrap["metric"] == metric)
                        & (bootstrap["resampling_scheme"] == preferred_scheme)
                        & (bootstrap["requested_block_length"] == cfg.bootstrap_block_days)
                    ]
                if bootstrap_row.empty:
                    mean_difference = lower_95 = upper_95 = float("nan")
                else:
                    selected = bootstrap_row.iloc[0]
                    mean_difference = float(selected["mean_difference"])
                    lower_95 = float(selected["lower_95"])
                    upper_95 = float(selected["upper_95"])
                row[f"{metric}_bootstrap_mean_difference"] = mean_difference
                row[f"{metric}_bootstrap_lower_95"] = lower_95
                row[f"{metric}_bootstrap_upper_95"] = upper_95
                row[f"{metric}_evidence"] = bootstrap_evidence_label(
                    mean_difference, lower_95, upper_95
                )

            accuracy_bootstrap = pd.DataFrame()
            if bootstrap is not None:
                preferred_scheme = (
                    "date_block"
                    if group["group_level"] == "pooled"
                    else "ticker_date_two_way"
                )
                accuracy_bootstrap = bootstrap.loc[
                    (bootstrap["cohort"] == bootstrap_cohort)
                    & (bootstrap["comparison"] == f"SigFlow v4 minus {baseline_name}")
                    & (bootstrap["metric"] == "within_20pct_accuracy")
                    & (bootstrap["resampling_scheme"] == preferred_scheme)
                    & (bootstrap["requested_block_length"] == cfg.bootstrap_block_days)
                ]
            if accuracy_bootstrap.empty:
                acc_mean = acc_lower = acc_upper = float("nan")
            else:
                selected_accuracy = accuracy_bootstrap.iloc[0]
                acc_mean = float(selected_accuracy["mean_difference"])
                acc_lower = float(selected_accuracy["lower_95"])
                acc_upper = float(selected_accuracy["upper_95"])
            row["primary_accuracy_bootstrap_mean_difference"] = acc_mean
            row["primary_accuracy_bootstrap_lower_95"] = acc_lower
            row["primary_accuracy_bootstrap_upper_95"] = acc_upper
            row["primary_accuracy_evidence"] = bootstrap_evidence_label(
                acc_mean, acc_lower, acc_upper, higher_is_better=True
            )
            row["primary_success_tolerance"] = cfg.primary_success_tolerance
            row["primary_accuracy_difference_percentage_points"] = row[
                f"accuracy_difference_{primary_label}pct_percentage_points"
            ]
            rows.append(row)
    return pd.DataFrame(rows)


def evaluate_predictions(
    predictions: pd.DataFrame,
    cfg: Config,
    data_source: str,
    evaluation_split: str,
) -> dict[str, object]:
    predictions = predictions.sort_values(
        ["origin_date", "evaluation_cohort", "ticker"]
    ).reset_index(drop=True)
    required_positive = [
        "actual_vol", "predicted_median_vol", "predicted_qlike_vol",
        "rolling_vol_baseline", "ewma_vol_baseline", "har_style_baseline",
    ]
    if predictions.duplicated(["ticker", "origin_date"]).any():
        raise ValueError("Every ticker/origin result must be unique.")
    values = predictions[required_positive].to_numpy(dtype=float)
    if not np.all(np.isfinite(values)) or np.any(values <= 0.0):
        raise ValueError("Inference requires finite, positive forecasts and outcomes.")

    point_table = evaluate_all_groups(predictions, cfg)
    accuracy_summary = build_accuracy_summary(predictions, cfg)
    daily_summary = build_daily_success_summary(predictions, cfg)
    probabilistic_summary = probabilistic_summary_by_group(predictions)
    raw_metrics, raw_matrix, raw_matrices = regime_summary_by_group(
        predictions, cfg, smoothed=False
    )
    smoothed_metrics, smoothed_matrix, smoothed_matrices = (
        regime_summary_by_group(predictions, cfg, smoothed=True)
    )
    non_overlapping = non_overlapping_phase_metrics(predictions, cfg)
    per_seed_metrics, seed_stability = build_per_seed_metrics(predictions, cfg)

    date_bootstrap = None
    ticker_date_bootstrap = None
    inference = None
    if cfg.run_block_bootstrap:
        date_tables = []
        groups = [("all", predictions)] + list(
            predictions.groupby("evaluation_cohort", sort=True)
        )
        for cohort, selected in groups:
            for block_days in cfg.bootstrap_block_sensitivity_days:
                repetitions = (
                    cfg.bootstrap_repetitions
                    if block_days == cfg.bootstrap_block_days
                    else cfg.bootstrap_sensitivity_repetitions
                )
                date_tables.append(paired_block_bootstrap(
                    selected,
                    cfg,
                    str(cohort),
                    block_days,
                    repetitions,
                ))
        date_bootstrap = pd.concat(date_tables, ignore_index=True)
        two_way_tables = [
            paired_ticker_date_bootstrap(selected, cfg, str(cohort))
            for cohort, selected in predictions.groupby(
                "evaluation_cohort", sort=True
            )
        ]
        ticker_date_bootstrap = pd.concat(two_way_tables, ignore_index=True)
        inference = apply_holm_correction(pd.concat(
            [date_bootstrap, ticker_date_bootstrap],
            ignore_index=True,
        ), cfg)
        date_bootstrap = inference[
            inference["resampling_scheme"] == "date_block"
        ].reset_index(drop=True)
        ticker_date_bootstrap = inference[
            inference["resampling_scheme"] == "ticker_date_two_way"
        ].reset_index(drop=True)

    baseline_comparison = build_baseline_comparison_summary(
        point_table, accuracy_summary, inference, cfg
    )
    primary_claim = build_primary_claim_summary(
        point_table,
        inference,
        seed_stability,
        predictions,
        cfg,
        data_source,
        evaluation_split,
    )

    detailed_results = predictions.drop(
        columns=["ticker_id", "split"],
        errors="ignore",
    )
    return {
        "detailed_test_results": detailed_results,
        "successful_forecasts": detailed_results[
            detailed_results["success_within_20pct"]
        ].reset_index(drop=True),
        "failed_forecasts": detailed_results[
            ~detailed_results["success_within_20pct"]
        ].reset_index(drop=True),
        "accuracy_summary": accuracy_summary,
        "daily_success_summary": daily_summary,
        "point_metrics": point_table,
        "baseline_comparison_summary": baseline_comparison,
        "probabilistic_summary": probabilistic_summary,
        "raw_regime_metrics": raw_metrics,
        "smoothed_regime_metrics": smoothed_metrics,
        "raw_confusion_matrix": raw_matrix,
        "smoothed_confusion_matrix": smoothed_matrix,
        "raw_confusion_matrices_by_cohort": raw_matrices,
        "smoothed_confusion_matrices_by_cohort": smoothed_matrices,
        "non_overlapping_phase_metrics": non_overlapping,
        "paired_bootstrap": date_bootstrap,
        "ticker_date_bootstrap": ticker_date_bootstrap,
        "inference_summary": inference,
        "per_seed_metrics": per_seed_metrics,
        "per_seed_stability": seed_stability,
        "primary_claim_summary": primary_claim,
    }



## Generalisation plots and result-only artifact saving

The notebook saves all detailed forecast values plus `training_summary.csv`, `baseline_comparison_summary.csv`, `per_seed_metrics.csv`, `per_seed_stability.csv`, `paired_block_bootstrap.csv`, `ticker_date_bootstrap.csv`, `inference_summary.csv`, and `primary_claim_summary.csv`. No raw OHLCV or model-input values are placed in result tables.


In [ ]:
def plot_confusion_matrix(
    matrix: np.ndarray,
    title: str,
) -> None:
    plt.figure(figsize=(5, 4))
    plt.imshow(matrix)
    plt.xticks(np.arange(3), REGIME_NAMES)
    plt.yticks(np.arange(3), REGIME_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    for row in range(3):
        for column in range(3):
            plt.text(column, row, str(matrix[row, column]), ha="center", va="center")
    plt.colorbar()
    plt.tight_layout()
    plt.show()


def plot_results(
    histories: list[dict[str, list[float]]],
    predictions: pd.DataFrame,
    evaluation: dict[str, object],
    cfg: Config,
) -> None:
    plt.figure(figsize=(10, 4))
    for model_index, history in enumerate(histories, start=1):
        epochs = np.arange(1, len(history["train_loss"]) + 1)
        plt.plot(
            epochs,
            history["train_loss"],
            alpha=0.55,
            label=f"Train {model_index}",
        )
        plt.plot(
            epochs,
            history["validation_loss"],
            label=f"Validation {model_index}",
        )
    plt.xlabel("Epoch")
    plt.ylabel("Objective")
    plt.title("Training and validation history")
    plt.legend()
    plt.tight_layout()
    plt.show()

    cohort_accuracy = evaluation["accuracy_summary"].query(
        "group_level == 'cohort'"
    ).copy()
    accuracy_columns = [
        "sigflow_within_5pct_accuracy",
        "sigflow_within_10pct_accuracy",
        "sigflow_within_20pct_accuracy",
        "sigflow_within_30pct_accuracy",
    ]
    x = np.arange(len(cohort_accuracy))
    width = 0.18
    plt.figure(figsize=(11, 4.5))
    for offset, column in enumerate(accuracy_columns):
        plt.bar(
            x + (offset - 1.5) * width,
            cohort_accuracy[column],
            width=width,
            label=column.replace("sigflow_within_", "≤").replace("_accuracy", ""),
        )
    plt.xticks(x, cohort_accuracy["cohort"], rotation=10)
    plt.ylim(0, 1)
    plt.ylabel("Forecast accuracy")
    plt.title("SigFlow accuracy by generalisation cohort")
    plt.legend()
    plt.tight_layout()
    plt.show()

    daily = evaluation["daily_success_summary"].copy()
    plt.figure(figsize=(13, 4.5))
    for cohort, selected in daily.groupby("cohort", sort=True):
        selected = selected.sort_values("origin_date")
        smoothed_accuracy = selected["success_rate_20pct"].rolling(
            20, min_periods=1
        ).mean()
        plt.plot(
            pd.to_datetime(selected["origin_date"]),
            smoothed_accuracy,
            label=f"{cohort} (20-origin rolling mean)",
        )
    plt.axhline(0.5, color="black", linestyle="--", alpha=0.4)
    plt.ylim(0, 1)
    plt.ylabel("Within-20% accuracy")
    plt.title("Success through unseen test days")
    plt.legend()
    plt.tight_layout()
    plt.show()

    if cfg.plot_each_ticker:
        for ticker in cfg.target_tickers:
            ticker_data = predictions[
                predictions["ticker"] == ticker
            ].sort_values("origin_date").copy()
            if ticker_data.empty:
                continue
            dates = pd.to_datetime(ticker_data["origin_date"])
            cohort = str(ticker_data["evaluation_cohort"].iloc[0])
            success = ticker_data["success_within_20pct"].to_numpy(dtype=bool)

            plt.figure(figsize=(13, 5))
            plt.plot(dates, ticker_data["actual_vol"], label="Actual future volatility")
            plt.plot(dates, ticker_data["predicted_median_vol"], label="SigFlow v4")
            plt.fill_between(
                dates,
                ticker_data["predicted_q05_vol"],
                ticker_data["predicted_q95_vol"],
                alpha=0.20,
                label="Calibration-tuned 90% interval",
            )
            plt.scatter(
                dates[success],
                ticker_data.loc[success, "actual_vol"],
                color="green",
                s=14,
                label="Within 20%",
            )
            plt.scatter(
                dates[~success],
                ticker_data.loc[~success, "actual_vol"],
                color="red",
                marker="x",
                s=18,
                label="Outside 20%",
            )
            plt.title(
                f"{ticker}: {cfg.horizon}-session volatility forecast | {cohort}"
            )
            plt.ylabel("Annualised volatility")
            plt.legend(ncol=2)
            plt.tight_layout()
            plt.show()

    plt.figure(figsize=(7, 4))
    plt.hist(predictions["pit"], bins=np.linspace(0, 1, 11))
    plt.axhline(len(predictions) / 10, linestyle="--")
    plt.title("Final-test PIT calibration")
    plt.xlabel("PIT")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    plot_confusion_matrix(
        evaluation["raw_confusion_matrix"],
        "Raw final-test regime confusion matrix",
    )
    plot_confusion_matrix(
        evaluation["smoothed_confusion_matrix"],
        "Past-only smoothed final-test regime confusion matrix",
    )


def save_results(
    models: list[RegimeFlowMixture],
    histories: list[dict[str, list[float]]],
    predictions: pd.DataFrame,
    evaluation: dict[str, object],
    dataset: MarketDataset,
    split: PreparedSplit,
    interval_scale: float,
    regime_temperature: float,
    cfg: Config,
) -> None:
    output_directory = Path(cfg.output_dir)
    output_directory.mkdir(parents=True, exist_ok=True)

    table_files = {
        "detailed_test_results.csv": evaluation["detailed_test_results"],
        "successful_test_forecasts.csv": evaluation["successful_forecasts"],
        "failed_test_forecasts.csv": evaluation["failed_forecasts"],
        "accuracy_summary.csv": evaluation["accuracy_summary"],
        "daily_success_summary.csv": evaluation["daily_success_summary"],
        "point_metrics.csv": evaluation["point_metrics"],
        "baseline_comparison_summary.csv": (
            evaluation["baseline_comparison_summary"]
        ),
        "primary_claim_summary.csv": evaluation["primary_claim_summary"],
        "per_seed_metrics.csv": evaluation["per_seed_metrics"],
        "per_seed_stability.csv": evaluation["per_seed_stability"],
        "training_summary.csv": evaluation["training_summary"],
        "probabilistic_summary.csv": evaluation["probabilistic_summary"],
        "raw_regime_metrics.csv": evaluation["raw_regime_metrics"],
        "smoothed_regime_metrics.csv": evaluation["smoothed_regime_metrics"],
        "non_overlapping_phase_metrics.csv": (
            evaluation["non_overlapping_phase_metrics"]
        ),
    }
    if evaluation["paired_bootstrap"] is not None:
        table_files["paired_block_bootstrap.csv"] = evaluation[
            "paired_bootstrap"
        ]
    if evaluation["ticker_date_bootstrap"] is not None:
        table_files["ticker_date_bootstrap.csv"] = evaluation[
            "ticker_date_bootstrap"
        ]
    if evaluation["inference_summary"] is not None:
        table_files["inference_summary.csv"] = evaluation[
            "inference_summary"
        ]
    for filename, table in table_files.items():
        table.to_csv(output_directory / filename, index=False)

    if cfg.save_audit_artifacts:
        dataset.data_manifest.to_csv(
            output_directory / "data_manifest.csv", index=False
        )
        dataset.skipped_samples.to_csv(
            output_directory / "skipped_samples.csv", index=False
        )
        split.leakage_audit.to_csv(
            output_directory / "leakage_audit.csv", index=False
        )

    for model_index, history in enumerate(histories, start=1):
        pd.DataFrame(history).to_csv(
            output_directory / f"training_history_model_{model_index}.csv",
            index=False,
        )

    torch.save({
        "model_state_dicts": [model.state_dict() for model in models],
        "config": asdict(cfg),
        "training_tickers": cfg.training_tickers,
        "unseen_test_tickers": cfg.unseen_test_tickers,
        "canonical_feature_names": dataset.feature_names,
        "active_feature_indices": split.active_feature_indices,
        "active_feature_names": split.active_feature_names,
        "dropped_feature_names": split.dropped_feature_names,
        "context_feature_names": split.context_feature_names,
        "data_source": dataset.data_source,
        "available_market_symbols": dataset.available_market_symbols,
        "winsor_lower": split.standardiser.lower,
        "winsor_upper": split.standardiser.upper,
        "feature_mean": split.standardiser.mean,
        "feature_scale": split.standardiser.scale,
        "ticker_thresholds": split.ticker_thresholds,
        "class_weights": split.class_weights,
        "train_regime_proportions": split.train_regime_proportions,
        "transition_matrices": split.transition_matrices,
        "initial_regime_probabilities": split.initial_regime_probabilities,
        "interval_scale": interval_scale,
        "regime_temperature": regime_temperature,
        "split_cutoffs": {
            "train": str(split.train_cutoff),
            "validation": str(split.validation_cutoff),
            "calibration": str(split.calibration_cutoff),
        },
    }, output_directory / "sigflow_v4_ensemble.pt")

    with open(
        output_directory / "run_config.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump({
            **asdict(cfg),
            "target_tickers": list(cfg.target_tickers),
            "output_dir": cfg.output_dir,
            "actual_data_source": dataset.data_source,
            "available_market_symbols": list(dataset.available_market_symbols),
            "active_feature_names": split.active_feature_names,
            "dropped_feature_names": split.dropped_feature_names,
            "interval_scale": interval_scale,
            "regime_temperature": regime_temperature,
        }, file, indent=2)

    print("\nSaved result artifacts to:", output_directory.resolve())


## Final-test discipline

The final evaluation contains only dates after the calibration cutoff:

- `seen_ticker_unseen_day`: trained tickers evaluated on future dates;
- `unseen_ticker_unseen_day`: completely held-out tickers evaluated on the identical future-date calendar.

The 22 training and 22 unseen tickers contain two names from each of 11 sectors. Held-out tickers never enter training, checkpoint selection, preprocessing, calibration, regime boundaries, transition priors, or HAR fitting. Optional feature selection uses three expanding inner folds that end no later than the canonical training cutoff.

The prespecified confirmatory claim is unseen-ticker QLIKE versus the transferable HAR baseline. Support requires real final-test data, the configured minimum of common origins, at least a 5% relative QLIKE improvement, a two-way-bootstrap 95% interval wholly below zero, consistent seed direction, and favourable effect direction at every prespecified date-block length. Pooled, sector, individual-day, and alternative-block results remain descriptive or robustness evidence.

Adding a modelling decision after inspecting final-test results turns this test into development evidence; create a later holdout period before making another final claim.


## Reading the result tables

`detailed_test_results.csv` contains one result row per ticker and forecast origin. It includes the forecast window, sector, actual volatility, median and QLIKE-optimal forecasts, quantiles, per-seed forecasts, baselines, errors, 5/10/20/30/50% accuracy flags, interval coverage, relative-volatility regimes, CRPS, PIT, and NLL.

A “successful day” means the median forecast issued on `origin_date` is within the configured 20% tolerance for realised volatility over `target_start_date` through `target_end_date`. It is not a one-session target and individual days are not significance tests.

`primary_claim_summary.csv` is the concise verdict. `inference_summary.csv` contains all formal and secondary comparisons, while the two bootstrap files expose block-length and ticker-population sensitivity. Synthetic runs are always labelled `pipeline_validation_only` regardless of their numerical scores.


# Run the configured watertight experiment


In [ ]:
def display_scrollable_results(
    table: pd.DataFrame,
    title: str,
    show_every_row: bool,
    height_px: int = 620,
) -> None:
    display(HTML(f"<h3>{title}</h3>"))
    if not show_every_row:
        display(table.head(100))
        print(
            f"Showing the first {min(100, len(table)):,} of {len(table):,} rows."
        )
        return
    rendered = table.to_html(
        index=False,
        border=0,
        float_format=lambda value: f"{value:.6g}",
    )
    display(HTML(
        f'<div style="max-height:{height_px}px; overflow:auto; '
        'border:1px solid #bbb; padding:4px">'
        f"{rendered}</div>"
    ))
    print(f"Rendered all {len(table):,} result rows in the scrollable table.")


def attach_run_identifiers(
    evaluation: dict[str, object],
    cfg: Config,
    evaluation_split: str,
) -> None:
    for value in evaluation.values():
        if isinstance(value, pd.DataFrame):
            if "run_name" not in value.columns:
                value.insert(0, "run_name", cfg.run_name)
            if "evaluation_split" not in value.columns:
                value.insert(1, "evaluation_split", evaluation_split)


def main(
    cfg: Config = CFG,
    dataset: MarketDataset | None = None,
    evaluation_split: str = "test",
    show_plots: bool = True,
    display_tables: bool = True,
    save_artifacts: bool = True,
) -> dict[str, object]:
    validate_config(cfg)
    if evaluation_split not in {"test", "validation"}:
        raise ValueError("evaluation_split must be test or validation.")

    print("=" * 78, flush=True)
    print(f"Starting SigFlow-Sim v4 | run={cfg.run_name}", flush=True)
    print("Mode:", cfg.experiment_mode, flush=True)
    print("Evaluation:", evaluation_split, flush=True)
    print("=" * 78, flush=True)
    if DEVICE.type == "cuda":
        print("CUDA device:", torch.cuda.get_device_name(0), flush=True)
    elif not cfg.quick_mode:
        print(
            "WARNING: CUDA is unavailable; this real profile will run on CPU "
            "and can take substantially longer.",
            flush=True,
        )

    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision(cfg.matmul_precision)
    set_seed(cfg.seed)

    if dataset is None:
        dataset = build_market_dataset(cfg)
    split = prepare_split(dataset, cfg)
    print(
        "Prepared rows | "
        f"train={len(split.train_indices):,}, "
        f"validation={len(split.validation_indices):,}, "
        f"calibration={len(split.calibration_indices):,}, "
        f"test={len(split.test_indices):,}",
        flush=True,
    )
    har_predictions = fit_ticker_specific_har(dataset, split, cfg)
    models, histories, best_epochs = fit_ensemble(dataset, split, cfg)
    print("\nBest validation epochs:", best_epochs)

    if evaluation_split == "test":
        interval_scale = 1.0
        regime_temperature = 1.0
        if cfg.calibrate_intervals or cfg.calibrate_regime_probabilities:
            print(
                "\nGenerating calibration forecasts "
                "(seen tickers only; never used for model fitting)..."
            )
            calibration_outputs = collect_ensemble_outputs(
                models,
                dataset,
                split,
                split.calibration_indices,
                cfg,
            )
            if cfg.calibrate_intervals:
                interval_scale = calibrate_interval_scale(
                    calibration_outputs["log_samples"],
                    dataset.targets_log_vol[split.calibration_indices],
                    cfg,
                )
            if cfg.calibrate_regime_probabilities:
                regime_temperature = calibrate_regime_temperature(
                    calibration_outputs["model_logits"],
                    split.regime_labels[split.calibration_indices],
                    cfg,
                )

        print(
            "\nGenerating both untouched final-test cohorts: "
            "new days, and new tickers plus new days..."
        )
        evaluation_indices = split.test_indices
        cohort_override = None
    else:
        # Development ablations use no calibration and never inspect test rows.
        interval_scale = 1.0
        regime_temperature = 1.0
        evaluation_indices = split.validation_indices
        cohort_override = "development_validation_seen_tickers"
        print("\nGenerating development-validation forecasts only...")

    outputs = collect_ensemble_outputs(
        models,
        dataset,
        split,
        evaluation_indices,
        cfg,
    )
    predictions = build_prediction_frame(
        outputs,
        dataset,
        split,
        evaluation_indices,
        interval_scale,
        regime_temperature,
        har_predictions,
        cfg,
        cohort_override=cohort_override,
    )
    evaluation = evaluate_predictions(
        predictions,
        cfg,
        data_source=dataset.data_source,
        evaluation_split=evaluation_split,
    )
    evaluation["training_summary"] = build_training_summary(
        histories, best_epochs, cfg
    )
    attach_run_identifiers(evaluation, cfg, evaluation_split)

    print("\nPrespecified primary claim:")
    print(evaluation["primary_claim_summary"].round(5).to_string(index=False))
    print("\nPer-seed stability:")
    print(evaluation["per_seed_stability"].round(5).to_string(index=False))
    print("\nTraining runtime and checkpoint summary:")
    print(evaluation["training_summary"].round(5).to_string(index=False))
    print("\nSigFlow point metrics (pooled, cohort, sector, and ticker):")
    print(
        evaluation["point_metrics"].query(
            "model == 'SigFlow v4'"
        ).round(5).to_string(index=False)
    )
    print("\nMultiple accuracy definitions:")
    print(
        evaluation["accuracy_summary"].round(5).to_string(index=False)
    )
    print(
        "\nBaseline comparison scorecard "
        "(negative error differences favour SigFlow):"
    )
    print(
        evaluation["baseline_comparison_summary"].round(5).to_string(index=False)
    )
    print("\nProbabilistic accuracy and interval coverage:")
    print(
        evaluation["probabilistic_summary"].round(5).to_string(index=False)
    )
    print("\nRaw regime accuracy:")
    print(
        evaluation["raw_regime_metrics"].round(5).to_string(index=False)
    )
    print("\nPast-only smoothed regime accuracy:")
    print(
        evaluation["smoothed_regime_metrics"].round(5).to_string(index=False)
    )
    if evaluation["paired_bootstrap"] is not None:
        print(
            "\nPaired date-block bootstrap differences "
            "(negative favours SigFlow):"
        )
        print(
            evaluation["paired_bootstrap"].round(5).to_string(index=False)
        )

    print("\nActual data source:", dataset.data_source)
    if dataset.data_source.startswith("synthetic"):
        print("WARNING: synthetic results validate the pipeline only.")

    if save_artifacts:
        save_results(
            models,
            histories,
            predictions,
            evaluation,
            dataset,
            split,
            interval_scale,
            regime_temperature,
            cfg,
        )
    if show_plots:
        plot_results(histories, predictions, evaluation, cfg)

    if display_tables:
        cohort_rows = evaluation["accuracy_summary"].query(
            "group_level in ['cohort', 'cohort_ticker']"
        )
        display_scrollable_results(
            cohort_rows,
            "Accuracy by unseen-data cohort and ticker",
            show_every_row=True,
            height_px=520,
        )
        display_scrollable_results(
            evaluation["daily_success_summary"],
            "Which forecast-origin days succeeded within 20%",
            show_every_row=True,
            height_px=620,
        )
        display_scrollable_results(
            evaluation["detailed_test_results"],
            "Every test result value (result fields only; no raw OHLCV/features)",
            show_every_row=(
                cfg.display_all_test_results and evaluation_split == "test"
            ),
            height_px=720,
        )

    print("\nComplete.")
    return {
        "dataset": dataset,
        "split": split,
        "models": models,
        "histories": histories,
        "best_epochs": best_epochs,
        "predictions": predictions,
        "evaluation": evaluation,
        "interval_scale": interval_scale,
        "regime_temperature": regime_temperature,
        "evaluation_split": evaluation_split,
    }


def make_ablation_config(
    base_cfg: Config,
    name: str,
    overrides: dict[str, object],
) -> Config:
    updates = dict(overrides)
    if "disabled_feature_groups" in updates:
        updates["disabled_feature_groups"] = tuple(dict.fromkeys(
            base_cfg.disabled_feature_groups
            + tuple(updates["disabled_feature_groups"])
        ))
    if "disabled_feature_names" in updates:
        updates["disabled_feature_names"] = tuple(dict.fromkeys(
            base_cfg.disabled_feature_names
            + tuple(updates["disabled_feature_names"])
        ))
    updates["run_name"] = f"{base_cfg.profile_name}_ablation_{name}"
    updates["run_block_bootstrap"] = False
    if base_cfg.ablation_reduced_budget:
        reduced_epochs = min(base_cfg.epochs, base_cfg.ablation_epochs)
        reduced_patience = min(base_cfg.patience, base_cfg.ablation_patience)
        updates.update({
            "epochs": reduced_epochs,
            "minimum_epochs": min(base_cfg.minimum_epochs, reduced_epochs),
            "patience": reduced_patience,
            "scheduler_patience": min(
                base_cfg.scheduler_patience,
                max(1, reduced_patience // 2),
            ),
            "ensemble_seeds": base_cfg.ablation_ensemble_seeds,
            "prediction_samples": min(256, base_cfg.prediction_samples),
        })
    return replace(base_cfg, **updates)


def inner_validation_fold_configs(
    base_cfg: Config,
) -> list[Config]:
    heldout_fraction = (
        base_cfg.train_fraction
        * base_cfg.inner_validation_fraction_of_training
    )
    fold_width = heldout_fraction / base_cfg.inner_validation_folds
    initial_train_fraction = base_cfg.train_fraction - heldout_fraction
    folds = []
    for fold_index in range(base_cfg.inner_validation_folds):
        fold_train_fraction = initial_train_fraction + fold_index * fold_width
        fold_cfg = replace(
            base_cfg,
            train_fraction=fold_train_fraction,
            validation_fraction=fold_width,
            run_name=f"{base_cfg.run_name}_inner_fold_{fold_index + 1}",
        )
        validate_config(fold_cfg)
        if (
            fold_cfg.train_fraction + fold_cfg.validation_fraction
            > base_cfg.train_fraction + 1e-12
        ):
            raise AssertionError("Inner validation crossed outer training cutoff.")
        folds.append(fold_cfg)
    return folds


def summarise_validation_ablation(
    name: str,
    description: str,
    fold_index: int,
    result: dict[str, object],
) -> dict[str, object]:
    point = result["evaluation"]["point_metrics"].query(
        "group_level == 'pooled' and model == 'SigFlow v4'"
    ).iloc[0]
    accuracy = result["evaluation"]["accuracy_summary"].query(
        "group_level == 'pooled'"
    ).iloc[0]
    return {
        "ablation": name,
        "description": description,
        "inner_fold": fold_index,
        "fold_train_cutoff": result["split"].train_cutoff,
        "fold_validation_cutoff": result["split"].validation_cutoff,
        "active_engineered_features": len(
            result["split"].active_feature_names
        ),
        "context_dimension": result["split"].context.shape[1],
        "validation_mae": float(point["mae"]),
        "validation_rmse": float(point["rmse"]),
        "validation_mape_percent": float(point["mape_percent"]),
        "validation_qlike": float(point["qlike"]),
        "validation_within_10pct_accuracy": float(
            accuracy["sigflow_within_10pct_accuracy"]
        ),
        "validation_within_20pct_accuracy": float(
            accuracy["sigflow_within_20pct_accuracy"]
        ),
        "validation_within_30pct_accuracy": float(
            accuracy["sigflow_within_30pct_accuracy"]
        ),
        "validation_raw_regime_accuracy": float(
            accuracy["raw_regime_accuracy"]
        ),
    }


def run_configured_validation_ablations(
    base_cfg: Config,
    dataset: MarketDataset,
) -> pd.DataFrame:
    jobs: list[tuple[str, str, dict[str, object]]] = [
        ("full_reference", "Full configured feature and objective reference.", {})
    ]

    if RUN_GROUP_ABLATIONS:
        unknown = sorted(set(SELECTED_GROUP_ABLATIONS) - set(GROUP_ABLATIONS))
        if unknown:
            raise ValueError(
                f"Unknown selected group ablations: {unknown}. "
                f"Valid names: {sorted(GROUP_ABLATIONS)}"
            )
        for name in SELECTED_GROUP_ABLATIONS:
            spec = GROUP_ABLATIONS[name]
            jobs.append((spec.name, spec.description, dict(spec.overrides)))

    if RUN_INDIVIDUAL_FEATURE_ABLATIONS:
        _, base_active_names, _ = resolve_active_feature_set(dataset, base_cfg)
        feature_names = (
            INDIVIDUAL_FEATURES_TO_ABLATE
            if INDIVIDUAL_FEATURES_TO_ABLATE
            else tuple(base_active_names)
        )
        unknown_features = sorted(set(feature_names) - set(dataset.feature_names))
        if unknown_features:
            raise ValueError(
                f"Unknown individual feature ablations: {unknown_features}"
            )
        already_inactive = sorted(set(feature_names) - set(base_active_names))
        if already_inactive:
            raise ValueError(
                "Individual ablations must name currently active features; "
                f"already inactive: {already_inactive}"
            )
        for feature_name in feature_names:
            safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", feature_name)
            jobs.append((
                f"without_{safe_name}",
                f"Remove only canonical feature {feature_name}.",
                {"disabled_feature_names": (feature_name,)},
            ))

    fold_count = base_cfg.inner_validation_folds
    print(
        f"\nRunning {len(jobs):,} configurations across {fold_count} "
        "expanding inner-validation folds. Every fold ends at or before the "
        "canonical training cutoff; outer validation and final test are not "
        "forecast in this selection loop."
    )
    summaries = []
    for job_index, (name, description, overrides) in enumerate(jobs, start=1):
        print(f"\nAblation {job_index}/{len(jobs)}: {name} — {description}")
        ablation_cfg = make_ablation_config(base_cfg, name, overrides)
        fold_configs = inner_validation_fold_configs(ablation_cfg)
        for fold_index, fold_cfg in enumerate(fold_configs, start=1):
            print(
                f"  Inner fold {fold_index}/{len(fold_configs)} | "
                f"train fraction={fold_cfg.train_fraction:.3f} | "
                f"validation fraction={fold_cfg.validation_fraction:.3f}"
            )
            result = main(
                fold_cfg,
                dataset=dataset,
                evaluation_split="validation",
                show_plots=False,
                display_tables=False,
                save_artifacts=False,
            )
            summaries.append(summarise_validation_ablation(
                name, description, fold_index, result
            ))
            del result
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    detailed = pd.DataFrame(summaries)
    metric_columns = [
        "validation_mae",
        "validation_rmse",
        "validation_mape_percent",
        "validation_qlike",
        "validation_within_10pct_accuracy",
        "validation_within_20pct_accuracy",
        "validation_within_30pct_accuracy",
        "validation_raw_regime_accuracy",
    ]
    identity_columns = [
        "ablation", "description", "active_engineered_features",
        "context_dimension",
    ]
    aggregate = detailed.groupby(
        identity_columns, as_index=False, dropna=False
    )[metric_columns].agg(["mean", "std"])
    aggregate.columns = [
        "_".join(str(part) for part in column if str(part))
        if isinstance(column, tuple) else str(column)
        for column in aggregate.columns
    ]
    aggregate["inner_fold_count"] = fold_count
    aggregate = aggregate.sort_values(
        ["validation_qlike_mean", "validation_mae_mean"]
    ).reset_index(drop=True)

    output_directory = Path(base_cfg.output_dir)
    output_directory.mkdir(parents=True, exist_ok=True)
    detailed.to_csv(
        output_directory / "development_inner_fold_ablation_details.csv",
        index=False,
    )
    aggregate.to_csv(
        output_directory / "development_validation_ablation_summary.csv",
        index=False,
    )
    display_scrollable_results(
        aggregate,
        "Inner-fold ablation comparison ranked by mean validation QLIKE",
        show_every_row=True,
        height_px=620,
    )
    return aggregate


# Build the canonical data once. Every configured feature ablation reuses the
# same rows; only the training-only model-input mask changes.
SHARED_DATASET = build_market_dataset(CFG)

ABLATION_SUMMARY = pd.DataFrame()
if RUN_GROUP_ABLATIONS or RUN_INDIVIDUAL_FEATURE_ABLATIONS:
    ABLATION_SUMMARY = run_configured_validation_ablations(
        CFG,
        SHARED_DATASET,
    )

# The prespecified base configuration is evaluated on the final cohorts only
# after optional development ablations have finished.
RESULTS = main(CFG, dataset=SHARED_DATASET)
